# FoodScholar — Quickstart

Three cells from zero to a populated graph using your local corpus + pre-computed NER/NEL.

```
pip install -e '.[elastic,neo4j,ontology]'
# local services (matching the inline config below):
docker run -d -p 9200:9200 -e discovery.type=single-node \
    -e xpack.security.enabled=false elasticsearch:8.13.0
docker run -d -p 7687:7687 -p 7474:7474 \
    -e NEO4J_AUTH=neo4j/change-me neo4j:5
```

This notebook does **not** run GLiNER. Pre-computed annotations from
`data/foodscholar/ner/*.csv` are loaded and attached to the chunks — fast,
deterministic, no model downloads. The "Under the hood" appendix at the
bottom shows how to do the same end-to-end with GLiNER + HNSW when no
pre-computed output is on disk.

## 1. Configure

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from foodscholar import FoodScholar
from foodscholar.logging import configure_logging
configure_logging(level="INFO")

CORPUS_DIR = REPO_ROOT / "data" / "foodscholar" / "corpus"   # chunks_*.csv
NEL_DIR    = REPO_ROOT / "data" / "foodscholar" / "ner"      # nel_chunks_*.csv
FOODON_OWL = REPO_ROOT / "data" / "foodon.owl"

fs = FoodScholar.from_config({
    "corpus": {
        "chunks_path": str(CORPUS_DIR),
        "annotated_snapshot_path": str(REPO_ROOT / "data" / "annotated.parquet"),
    },
    "ontology": {
        "foodon_path": str(FOODON_OWL),
        "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
        "prefix_filter": ["FOODON:"],
    },
    "storage": {
        "chunk_store": {
            "backend": "elastic",
            "url": "http://localhost:9200",
            "index": "foodscholar_chunks",
        },
        "graph_store": {
            "backend": "neo4j",
            "url": "bolt://localhost:7687",
            "user": "neo4j",
            "password": "password",
        },
    },
    "layer_a": {
        # Filter generic mentions the linker dumps onto the umbrella term
        # FOODON:00001002 ('food product'). Without these, ~426 direct hits keep
        # its direct_share at 0.102 (just above the 0.10 umbrella cutoff), so it
        # survives as a 2.5k-chunk dumping ground. Removing them drops
        # direct_share to ~0.03 -> the umbrella rule fires -> those chunks lift
        # to specific descendant shelves. (Keeps the default 'fish' entry too.)
        "link_blocklist": [
            {"surface": "fish", "ontology_id": "FOODON:00002281"},
            {"surface": "foods", "ontology_id": "FOODON:00001002"},
            {"surface": "food products", "ontology_id": "FOODON:00001002"},
            {"surface": "food product", "ontology_id": "FOODON:00001002"},
            {"surface": "whole foods", "ontology_id": "FOODON:00001002"},
            {"surface": "perishable foods", "ontology_id": "FOODON:00001002"},
            {"surface": "perishable products", "ontology_id": "FOODON:00001002"},
            {"surface": "foods and beverages", "ontology_id": "FOODON:00001002"},
            {"surface": "food and beverages", "ontology_id": "FOODON:00001002"},
            {"surface": "certain foods", "ontology_id": "FOODON:00001002"},
            {"surface": "specific food", "ontology_id": "FOODON:00001002"},
            {"surface": "real food", "ontology_id": "FOODON:00001002"},
            {"surface": "superfoods", "ontology_id": "FOODON:00001002"},
            {"surface": "imported foods", "ontology_id": "FOODON:00001002"},
            {"surface": "competitive foods", "ontology_id": "FOODON:00001002"},
            {"surface": "local foods", "ontology_id": "FOODON:00001002"},
        ],
    },
})
fs.info()

{'foodscholar': '0.1.0',
 'config_hash': '4982fd8cd2e37150',
 'chunk_store': 'elastic',
 'graph_store': 'neo4j',
 'embedder': 'lazy(BAAI/bge-base-en-v1.5)',
 'llm': 'mock-llm-v0',
 'ontology': 'configured',
 'ner': 'gliner',
 'nel_backend': 'hnsw',
 'prompt_version': 'v1'}

## 2. Init stores & ingest

`fs.init()` creates the Elastic index and the Neo4j unique constraints — idempotent.

`fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR)` reads every `chunks_*.csv`, attaches the matching annotations from `nel_*.csv`, and upserts to Elastic. **No GLiNER, no HNSW, no embedding** — this is fast and works without the `[annotate]` extra installed.

Chunk-text embeddings are populated in a separate step (§3) so you can iterate on annotations without re-encoding.

In [3]:
fs.init()
# Abstracts are excluded by design — the prototype's `nel_*.csv` set has no
# nel_*abstracts*.csv file, so a chunk loaded from chunks_abstracts.csv would
# arrive with zero NEL annotations. Adding 600k+ such chunks would inflate the
# corpus pool without contributing any support to Layer A. Restore them only
# after the abstract NEL pass is re-run (or after switching to GLiNER+HNSW
# via `fs.load_and_annotate(CORPUS_DIR)`).
meta = fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR, ignore_source_types=['abstract'])
print(meta)

HEAD http://localhost:9200/foodscholar_chunks [status:200 duration:0.006s]
HEAD http://localhost:9200/foodscholar_chunks_entities [status:200 duration:0.002s]
Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT shelf_id IF NOT EXISTS FOR (e:Shelf) REQUIRE (e.shelf_id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT shelf_id FOR (e:Shelf) REQUIRE (e.shelf_id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT shelf_id IF NOT EXISTS FOR (s:Shelf) REQUIRE s.shelf_id IS UNIQUE'
Received n

artifact_id='ingest-10b8ee4fc24f' phase='ingest' config_hash='4982fd8cd2e37150' upstream_artifact_ids=[] record_count=13344 schema_version='v0.1' created_at=datetime.datetime(2026, 5, 26, 18, 30, 30, 603337, tzinfo=datetime.timezone.utc)


## 3. Embed (optional, for vector search)

Run this once chunks are in the store and you want kNN search to work. It builds the
production embedder — `SourceTypeRouter(SPECTER2 for abstracts, BGE-large for guides /
textbooks)` — lazily on the first call (~1.7 GB model load) and patches each chunk's
`embedding` + `embedding_model` fields in Elastic via a scoped `_update` (annotations untouched).

Default `only_missing=True` skips chunks that already carry a real vector, so re-runs after
adding new chunks only encode what's new.

Requires the `[annotate]` extra (`pip install -e '.[annotate]'`).

In [2]:
meta = fs.embed()
print(meta)

/mnt/miniconda3/envs/foodscholar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No device provided, using cpu
Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.019s]
Batches: 100%|██████████| 2/2 [00:08<00:00,  4.11s/it]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.174s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.011s]
Batches: 100%|██████████| 2/2 [00:07<00:00,  4.00s/it]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.354s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.012s]
Batches: 100%|██████████| 2/2 [00:05<00:00,  2.57s/it]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.123s]
PO

artifact_id='embed-0213d4fe3730' phase='embed' config_hash='4982fd8cd2e37150' upstream_artifact_ids=[] record_count=13344 schema_version='v0.1' created_at=datetime.datetime(2026, 5, 26, 18, 52, 59, 819828, tzinfo=datetime.timezone.utc)


## 4. Build & explore entities

Linked entities are first-class. `fs.build_entities()` walks the chunk store, dedupes every `EntityLink` by `ontology_id` (FOODON, CHEBI, GAZ, ...), aggregates `mention_count` / `chunk_count` / sample `chunk_ids`, enriches FoodOn ids with label + synonyms + ancestors from the loaded ontology, and writes everything to:

- **Elastic** — a dedicated `foodscholar_chunks_entities` index for fast lookup + search.
- **Neo4j** — `(:Entity)` nodes plus `(:Chunk)-[:MENTIONS {confidence, method}]->(:Entity)` edges.

Then `fs.entities` is the user-facing handle: `.list(prefix="FOODON")`, `.get(id)`, `.search("olive")`, `.chunks_for(id)`.

In [3]:
meta = fs.build_entities()
print(meta)



POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.046s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.039s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.041s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.040s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.041s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.036s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.037s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.036s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.061s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.038s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.049s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.057s]
POST

artifact_id='build_entities-6b1e906d085a' phase='build_entities' config_hash='4982fd8cd2e37150' upstream_artifact_ids=[] record_count=3879 schema_version='v0.1' created_at=datetime.datetime(2026, 5, 26, 18, 54, 28, 382382, tzinfo=datetime.timezone.utc)


In [4]:
print(f"\ntotal entities: {len(fs.entities)}")
# Top FoodOn entities by chunk_count.
for ent in fs.entities.list(k=8):
    print(f"  {ent.ontology_id:24s}  {ent.label!r:40s}  chunks={ent.chunk_count}")

# Lexical search over labels + synonyms.
print("\nsearch 'olive':")
for ent in fs.entities.search("olive", k=3):
    print(f"  {ent.ontology_id}  {ent.label}")

# Reverse lookup: chunks that mention a specific entity.
top = fs.entities.list(prefix="FOODON", k=1)
if top:
    print(f"\nchunks mentioning {top[0].ontology_id} ({top[0].label}):")
    for c in fs.entities.chunks_for(top[0].ontology_id, k=3):
        print(f"  {c.chunk_id}  {c.text[:80]}")

POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.041s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.012s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.010s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.008s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.007s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.007s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.006s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.009s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 durat


total entities: 3879


POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.030s]
POST http://localhost:9200/foodscholar_chunks_entities/_search [status:200 duration:0.032s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.008s]


  FOODON:03317626           'processed meat-had'                      chunks=1
  FOODON:03306429           'meat'                                    chunks=407
  FOODON:03411222           'fish'                                    chunks=396
  FOODON:00004298           'poultry'                                 chunks=366
  FOODON:02020891           'milk'                                    chunks=522
  FOODON:03302992           'Animal cheese'                           chunks=1
  FOODON:03420194           'eggs'                                    chunks=270
  FOODON:00001874           'fortified foods'                         chunks=40

search 'olive':
  FOODON:03317509  olive
  FOODON:03301826  olive oil
  FOODON:03306633  Virgin olive oil

chunks mentioning FOODON:00003339 (calories):
  003ac0a7-18ca-4184-8bb8-cd1a20d7a7a1  For children: Intense sweeteners are safe, but not intended for infants and youn
  00c0735a-1a7c-47e1-a330-6d337fcc1852  Cooking foods in an iron skillet adds iron

## 5. Build Layer A (backbone)

Layer A is FoodOn (and the other OBO ontologies we link to) projected onto **your** corpus — not a copy of the raw ontology. For every FoodOn class, count how many chunks have at least one link to that class or any of its descendants, then prune ruthlessly: blacklist organizational classes, drop everything below `min_support`, lift shelves above `max_depth` to a closer ancestor, and collapse single-child chains into their leaves (recording the dropped ids in `see_also`).

Non-foods facets (`health`, `nutrients`, `dietary_patterns`, `allergies`) derive from `Mention.entity_type` on EntityLinks. The prototype's pre-computed NEL data sets every `entity_type` to `"other"`, so on this corpus those facets land as **stub roots only** — they'll fill in once chunks are re-annotated with GLiNER. Sustainability is always a stub root (no entity_type maps to it).

Layer A here is **projection only**. Chunk attachment (`fs.attach()`) is the next phase.

Re-running `fs.build_layer_a()` is idempotent: every call clears the prior `:Shelf` projection from the graph store before writing the new one, so you can re-tune `min_support` / `blacklist_terms` / `facet_overrides` freely without stale ghost shelves piling up across iterations.

In [5]:
# Tune for the current corpus — defaults are conservative.
fs.config.layer_a.min_support = 25
fs.config.layer_a.max_depth = 6
fs.config.layer_a.collapse_single_child_chains = True

# Blacklist two FoodOn organizational classes that surface as junk navigation
# roots on this corpus. Both are datum/umbrella terms, not foods anyone
# searches for — `food calorie datum` is a measurement datum (it kept showing
# up as a top entity but is a hierarchy leaf with no real children), and
# `edible food` is a near-root umbrella that absorbs everything below it.
# Blacklisting drops them *before* the threshold pass, so their chunks lift to
# the nearest surviving food ancestor instead of piling onto a useless shelf.
# Matching is case-insensitive against the FoodOn label; we append rather than
# replace so the default upper-ontology scaffolding stays blocked too.
fs.config.layer_a.blacklist_terms = fs.config.layer_a.blacklist_terms + [
    "food calorie datum",
    "edible food",
]

# Echo the resolved foods-facet config so the prune cascade's parameters
# are visible at a glance. Override anything per-facet via
# `fs.config.layer_a.facet_overrides["foods"] = FacetConfig(...)`.
_foods = fs.config.layer_a.resolve_facet('foods')
print(f'foods config: min_support={_foods.min_support}, max_depth={_foods.max_depth}, '
      f'collapse={_foods.collapse_single_child_chains}, '
      f'umbrella=({_foods.umbrella_direct_share_max}, {_foods.umbrella_lifted_share_min}, '
      f'min_count={_foods.umbrella_min_count}), '
      f'blacklist={_foods.blacklist_terms}')

meta = fs.build_layer_a()
print(meta)

ontology.cache_hit path=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon_cache.parquet


foods config: min_support=25, max_depth=6, collapse=True, umbrella=(0.1, 0.85, min_count=25), blacklist=['material entity', 'physical object', 'manufactured product', 'food calorie datum', 'edible food']


2026-05-26T18:57:05.908346Z [info     ] ontology.loaded                cached=True n_terms=39278 source=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon.owl
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.084s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.076s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.112s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.067s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.078s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.088s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.094s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.077s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.073s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.074s]
POST http

artifact_id='build-layer-a-6fc30586a339' phase='build-layer-a' config_hash='3e95e5b0e58fdda1' upstream_artifact_ids=[] record_count=232 schema_version='v0.1' created_at=datetime.datetime(2026, 5, 26, 18, 57, 33, 655307, tzinfo=datetime.timezone.utc)


In [ ]:
from collections import Counter

shelves = fs.graph_store.list_shelves()
by_facet = Counter(s.facet for s in shelves)
print('shelves per facet:')
for facet, n in sorted(by_facet.items(), key=lambda kv: -kv[1]):
    print(f'  {facet:18s} {n}')

foods = [s for s in shelves if s.facet == 'foods']
print(f'\ntop 10 foods shelves by chunk_count:')
for s in sorted(foods, key=lambda s: -s.chunk_count)[:10]:
    direct_share = s.support_direct / s.chunk_count if s.chunk_count else 0
    print(f'  {s.label[:40]:40s}  {s.chunk_count:>5d}  (direct={s.support_direct} lifted={s.support_lifted} direct_share={direct_share:.3f})')

print(f'\nbottom 5 surviving foods shelves:')
for s in sorted(foods, key=lambda s: s.chunk_count)[:5]:
    print(f'  {s.label[:40]:40s}  {s.chunk_count:>5d}')

depths = Counter(s.depth for s in foods)
print(f'\nfoods depth distribution:')
for d in sorted(depths):
    print(f'  depth {d}: {depths[d]}')

# Inflation diagnostic — shelves whose support is mostly lifted (per the spec,
# >0.9 means a deep dense subtree funneled into one ancestor). After the
# umbrella rule lands these should be rare — surviving inflated shelves are
# real navigation roots (e.g. `food product` itself) rather than scaffolding.
inflated = [s for s in foods if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9]
if inflated:
    print(f'\n{len(inflated)} potentially-inflated shelves (support_lifted/chunk_count > 0.9):')
    for s in sorted(inflated, key=lambda s: -s.chunk_count)[:5]:
        ratio = s.support_lifted / s.chunk_count
        direct_share = s.support_direct / s.chunk_count
        print(f'  {s.label[:40]:40s}  lifted_ratio={ratio:.2f}  direct_share={direct_share:.3f}  count={s.chunk_count}')
else:
    print('\nno inflation flags.')

# Orphan diagnostic — shelves at depth 0 that aren't the stub roots. After
# the umbrella rule lands these are the navigation roots that no surviving
# parent could be found for. Whitelisting the foods root (FOODON:00001002)
# typically reduces this to 1-2.
depth0_foods = [s for s in foods if s.depth == 0]
print(f'\n{len(depth0_foods)} foods shelves at depth 0 (roots):')
for s in sorted(depth0_foods, key=lambda s: -s.chunk_count)[:10]:
    print(f'  {s.label[:40]:40s}  count={s.chunk_count}  foodon_id={s.foodon_id}')

  [food            ] 'olive oil'  @34:43  (keyword-ner-v0)
  [food            ] 'whole'  @45:50  (keyword-ner-v0)


### 5b. Why these thresholds? — parameter sweep

The three knobs the prune cascade exposes (`min_support`, `umbrella_direct_share_max`, `umbrella_min_count`) are empirical, not derived. This cell sweeps each one independently against the live corpus and plots the trade-off curves, then a 2D heatmap of the two umbrella knobs together. The chosen values are marked.

Cheap: support is collected once over the chunk store, then `prune(...)` re-runs per config combo without touching Neo4j. Runs in seconds even on a multi-thousand-chunk corpus.

In [ ]:
import matplotlib.pyplot as plt
from foodscholar.layer_a.propagate import collect_support
from foodscholar.layer_a.prune import prune
from foodscholar.config import FacetConfig, LayerAConfig

# Collect support once — every sweep point re-prunes against this in-memory
# table. ~seconds for ~13k chunks; the bottleneck would be Neo4j writes,
# which we skip.
def _chunk_iter():
    for batch in fs.chunk_store.iter_chunks():
        yield from batch

base = fs.config.layer_a.resolve_facet('foods')
support = collect_support(_chunk_iter(), fs.ontology,
                          min_link_confidence=base.min_link_confidence,
                          facet='foods')
print(f'support table: {len(support.with_descendants)} terms, '
      f'{sum(support.direct.values())} direct mentions')

def _resolve_with(**overrides):
    """Build a fully-resolved foods FacetConfig by overlaying overrides on the
    live `fs.config.layer_a`. Goes through `resolve_facet` so we touch only the
    public config surface."""
    base_cfg = fs.config.layer_a
    layer_a = LayerAConfig.model_validate(base_cfg.model_dump())
    layer_a.facet_overrides = dict(layer_a.facet_overrides)
    layer_a.facet_overrides['foods'] = FacetConfig(**overrides)
    return layer_a.resolve_facet('foods')

def _summarize(shelves: list) -> dict:
    inflated = sum(1 for s in shelves
                   if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9)
    roots = sum(1 for s in shelves if s.parent_shelf_id is None)
    return {'n_shelves': len(shelves), 'n_inflated': inflated, 'n_roots': roots}

# Run the three single-knob sweeps.
ms_grid = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200]
ds_grid = [0.0, 0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
mc_grid = [10, 25, 50, 75, 100, 150, 200, 300, 500, 1000]

ms_data = [(ms, _summarize(prune(support, fs.ontology, _resolve_with(min_support=ms), 'foods'))) for ms in ms_grid]
ds_data = [(ds, _summarize(prune(support, fs.ontology, _resolve_with(umbrella_direct_share_max=ds), 'foods'))) for ds in ds_grid]
mc_data = [(mc, _summarize(prune(support, fs.ontology, _resolve_with(umbrella_min_count=mc), 'foods'))) for mc in mc_grid]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def _plot(ax, data, knob_name, chosen, log_x=False):
    xs = [d[0] for d in data]
    n_shelves = [d[1]['n_shelves'] for d in data]
    n_inflated = [d[1]['n_inflated'] for d in data]
    ax.plot(xs, n_shelves, 'o-', label='surviving shelves')
    ax.plot(xs, n_inflated, 's--', label='inflated (>0.9 lifted)')
    ax.axvline(chosen, linestyle=':', label=f'chosen = {chosen}')
    ax.set_title(knob_name)
    ax.set_xlabel(knob_name)
    ax.set_ylabel('shelves')
    ax.grid(True)
    if log_x:
        ax.set_xscale('log')
    ax.legend()

_plot(axes[0], ms_data, 'min_support', base.min_support, log_x=True)
_plot(axes[1], ds_data, 'umbrella_direct_share_max', base.umbrella_direct_share_max)
_plot(axes[2], mc_data, 'umbrella_min_count', base.umbrella_min_count, log_x=True)

fig.suptitle('Layer A — single-knob parameter sweep')
plt.tight_layout()
plt.show()

# Print the chosen-point summary as numeric grounding for the chart.
chosen_cfg = _resolve_with()  # all defaults
chosen_shelves = prune(support, fs.ontology, chosen_cfg, 'foods')
chosen = _summarize(chosen_shelves)
print(f"\nat chosen config (min_support={base.min_support}, "
      f"direct_share_max={base.umbrella_direct_share_max}, "
      f"min_count={base.umbrella_min_count}):")
print(f"  shelves:               {chosen['n_shelves']}")
print(f"  inflated (>0.9 lifted): {chosen['n_inflated']}")
print(f"  orphan roots:           {chosen['n_roots']}")

ontology.cache_hit path=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon_cache.parquet
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:51.461720Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=3 n_returned=3


c1: Mediterranean diet rich in olive oil reduces cardiovascular risk.
    [dietary_pattern ] 'Mediterranean diet'  @0:18
    [food            ] 'olive oil'  @27:36
    [biomarker       ] 'cardiovascular risk'  @45:64


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:51.988214Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


c2: Whole grain consumption is associated with lower mortality.
    [processing      ] 'Whole grain'  @0:11
    [biomarker       ] 'mortality'  @49:58


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:52.688701Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


...

c4: Dietary intake of quinoa improves glycemic control in adults with type 2 diabetes.
    [food            ] 'quinoa'  @18:24
    [biomarker       ] 'glycemic control'  @34:50
    [population      ] 'adults'  @54:60
    [health          ] 'type 2 diabetes'  @66:81


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:53.584325Z [warning  ] agent_ner.mention_not_in_text  mention='extra virgin'
2026-05-19T12:46:53.598713Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=3 n_returned=4


c5: Extra virgin olive oil contains polyphenols that reduce inflammation markers.
    [food            ] 'Extra virgin olive oil'  @0:22
    [nutrient        ] 'polyphenols'  @32:43
    [biomarker       ] 'inflammation markers'  @56:76


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:53.927564Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


c6: School-based peanut allergy education reduces accidental exposures among children.
    [allergen        ] 'peanut allergy'  @13:27
    [population      ] 'children'  @73:81


In [ ]:
import numpy as np

# Joint sweep: direct_share_max × min_count. Heatmap colors the joint effect.
ds_axis = [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30]
mc_axis = [25, 50, 75, 100, 150, 200, 300, 500]

grid_shelves = np.zeros((len(mc_axis), len(ds_axis)))
grid_inflated = np.zeros((len(mc_axis), len(ds_axis)))
for i, mc in enumerate(mc_axis):
    for j, ds in enumerate(ds_axis):
        cfg = _resolve_with(umbrella_direct_share_max=ds, umbrella_min_count=mc)
        shelves = prune(support, fs.ontology, cfg, 'foods')
        s = _summarize(shelves)
        grid_shelves[i, j] = s['n_shelves']
        grid_inflated[i, j] = s['n_inflated']
clean = grid_shelves - grid_inflated

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def _heatmap(ax, grid, cmap, title):
    im = ax.imshow(grid, aspect='auto', origin='lower', cmap=cmap)
    ax.set_xticks(range(len(ds_axis)))
    ax.set_xticklabels([f'{d:g}' for d in ds_axis])
    ax.set_yticks(range(len(mc_axis)))
    ax.set_yticklabels(mc_axis)
    ax.set_xlabel('umbrella_direct_share_max')
    ax.set_ylabel('umbrella_min_count')
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    for i in range(len(mc_axis)):
        for j in range(len(ds_axis)):
            ax.text(j, i, f'{grid[i, j]:.0f}', ha='center', va='center')

_heatmap(axes[0], clean, 'viridis', 'Clean shelves (surviving − inflated)')
_heatmap(axes[1], grid_inflated, 'Reds', 'Inflated shelves')

# Mark the chosen config in both heatmaps.
if base.umbrella_direct_share_max in ds_axis and base.umbrella_min_count in mc_axis:
    j_chosen = ds_axis.index(base.umbrella_direct_share_max)
    i_chosen = mc_axis.index(base.umbrella_min_count)
    for ax in axes:
        ax.add_patch(plt.Rectangle((j_chosen - 0.5, i_chosen - 0.5), 1, 1,
                                   fill=False, edgecolor='black', linewidth=2))

fig.suptitle('Layer A — joint sweep of the two umbrella knobs')
plt.tight_layout()
plt.show()

### 5c. What happens when you raise `max_depth`?

The depth cap doesn't drop deep terms — it **lifts** them to the nearest surviving ancestor at depth ≤ cap (see [prune.py](../src/foodscholar/layer_a/prune.py) and the §7.0 PROGRESS entry on op order). So raising the cap doesn't *add* terms; it lets terms that were already there land deeper instead of bunching up at the cap.

Three things should shift as the cap rises:

1. **Surviving shelves go up** — more terms keep their natural ontology depth instead of getting clipped + collapsed away.
2. **`support_lifted` redistributes downward** — chunks that were lifting onto a shallow ancestor now find a more specific match closer to the leaf. Inflation drops on shelves that were absorbing everything from below.
3. **Median direct-share rises** — deep shelves get their own chunks instead of letting them lift up.

This sweep is **projection-only** (same recipe as §5b): support is collected once, `prune()` re-runs per cap value, no Neo4j writes. Re-running `fs.attach()` per cap would also redistribute *edges*, but the projection-side picture is cheap and tells you whether raising the cap is even worth doing.

In [ ]:
from statistics import median

# Depth sweep — `support` and `_resolve_with` are still in scope from §5b.
depth_grid = [3, 4, 5, 6, 7, 8, 9]

def _depth_summary(shelves):
    if not shelves:
        return {'n_shelves': 0, 'n_inflated': 0, 'median_direct_share': 0.0,
                'depth_dist': {}, 'shelves_at_or_above_cap': 0}
    inflated = sum(1 for s in shelves
                   if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9)
    direct_shares = [s.support_direct / s.chunk_count
                     for s in shelves if s.chunk_count > 0]
    depth_dist = Counter(s.depth for s in shelves)
    cap = max(depth_dist) if depth_dist else 0
    return {
        'n_shelves': len(shelves),
        'n_inflated': inflated,
        'median_direct_share': median(direct_shares) if direct_shares else 0.0,
        'depth_dist': dict(depth_dist),
        'shelves_at_or_above_cap': depth_dist.get(cap, 0),
    }

depth_data = []
for d in depth_grid:
    cfg = _resolve_with(max_depth=d)
    shelves = prune(support, fs.ontology, cfg, 'foods')
    depth_data.append((d, _depth_summary(shelves)))

xs = [d[0] for d in depth_data]
n_shelves = [d[1]['n_shelves'] for d in depth_data]
n_inflated = [d[1]['n_inflated'] for d in depth_data]
median_share = [d[1]['median_direct_share'] for d in depth_data]

CHOSEN_DEPTH = fs.config.layer_a.max_depth

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: shelves + inflation vs depth cap.
ax = axes[0]
ax.plot(xs, n_shelves, 'o-', label='surviving shelves')
ax.plot(xs, n_inflated, 's--', label='inflated (>0.9 lifted)')
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('max_depth: surviving + inflated shelves')
ax.set_xlabel('max_depth')
ax.set_ylabel('shelves')
ax.grid(True)
ax.legend()

# Panel 2: median direct-share vs depth cap.
ax = axes[1]
ax.plot(xs, median_share, 'o-', label='median direct-share')
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('median direct-share')
ax.set_xlabel('max_depth')
ax.set_ylabel('median support_direct / chunk_count')
ax.set_ylim(0, 1.05)
ax.grid(True)
ax.legend()

# Panel 3: depth distribution as stacked bars.
ax = axes[2]
all_depths = sorted({d for _, s in depth_data for d in s['depth_dist']})
bottoms = [0] * len(xs)
for depth in all_depths:
    heights = [s[1]['depth_dist'].get(depth, 0) for s in depth_data]
    ax.bar(xs, heights, bottom=bottoms, label=f'depth {depth}')
    bottoms = [b + h for b, h in zip(bottoms, heights, strict=True)]
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('shelf count by projection depth')
ax.set_xlabel('max_depth')
ax.set_ylabel('shelves')
ax.grid(True)
ax.legend(title='projection depth', ncol=2)

fig.suptitle('Layer A — depth-cap sweep')
plt.tight_layout()
plt.show()

# Numeric grounding — print the table so the chart values are auditable.
print(f'\n{"depth":>5s}  {"shelves":>7s}  {"inflated":>8s}  {"med_direct":>10s}  '
      f'{"shelves_at_cap":>14s}')
for d, s in depth_data:
    mark = '  *' if d == CHOSEN_DEPTH else ''
    print(f'{d:>5d}  {s["n_shelves"]:>7d}  {s["n_inflated"]:>8d}  '
          f'{s["median_direct_share"]:>10.3f}  {s["shelves_at_or_above_cap"]:>14d}{mark}')

### 5d. Why does so much support land at the synthetic root? — NEL coverage audit

The synthetic `Foods` root carries `support_lifted = chunk_count` by construction — every chunk reaching the facet contributes through lifting since nothing in any ontology resolves to the root itself. But the *size* of that number relative to the total corpus is informative: a high root count means many chunks failed to find a specific shelf, either because they have no FOODON links at all, or because all their links got umbrella-pruned.

This cell walks the chunk store once and reports:

- **NEL coverage**: how many chunks have ≥1 `foodon_id`, how many have only non-FOODON `entity_links`, how many have no links at all.
- **Sample of zero-foodon chunks**: 5 random texts so you can eyeball whether they're nutrition-y-but-not-food (expected) or food-content-the-NEL-missed (a real coverage gap).

In [ ]:
import random
from collections import Counter

n_total = 0
n_with_foodon = 0
n_only_other_obo = 0  # has entity_links but zero FOODON ids
n_no_links = 0
prefix_counter: Counter = Counter()
zero_foodon_samples: list = []

for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        n_total += 1
        has_foodon = bool(chunk.foodon_ids)
        has_any_link = bool(chunk.entity_links)
        if has_foodon:
            n_with_foodon += 1
        elif has_any_link:
            n_only_other_obo += 1
            for link in chunk.entity_links:
                oid = link.ontology_id
                prefix = oid.split(':', 1)[0] if ':' in oid else '(none)'
                prefix_counter[prefix] += 1
        else:
            n_no_links += 1

        # Keep a reservoir sample of zero-foodon chunks for hand inspection.
        if not has_foodon and len(zero_foodon_samples) < 200:
            zero_foodon_samples.append(chunk)

print(f'chunks total:                  {n_total:>6d}  (100.0%)')
print(f'  ≥1 FOODON id (well-placed):  {n_with_foodon:>6d}  ({100*n_with_foodon/n_total:>5.1f}%)')
print(f'  only non-FOODON entity_links: {n_only_other_obo:>6d}  ({100*n_only_other_obo/n_total:>5.1f}%)')
print(f'  zero entity links at all:     {n_no_links:>6d}  ({100*n_no_links/n_total:>5.1f}%)')

print(f'\nontology prefixes seen on chunks WITHOUT a FOODON id ({len(prefix_counter)} distinct):')
for prefix, n in prefix_counter.most_common(10):
    print(f'  {prefix:15s}  {n:>6d}')

print(f'\n5 random zero-foodon chunk texts (hand-check whether they should have had a FOODON link):')
random.seed(0)
for chunk in random.sample(zero_foodon_samples, k=min(5, len(zero_foodon_samples))):
    has_other = bool(chunk.entity_links)
    marker = '  [has non-FOODON links]' if has_other else '  [no links at all]'
    print(f'\n  {chunk.chunk_id}{marker}')
    print(f'    {chunk.text[:240].replace(chr(10), " ")}{"..." if len(chunk.text) > 240 else ""}')

### 5e. Sanity gate — 50 random FOODON-linked chunks

Pick 50 random chunks that have ≥1 FOODON link and print the text + the projected shelf labels each one rolls up to. The shelves shown mirror what `fs.attach()` writes in §6 — for every `foodon_id` on the chunk, this finds the deepest surviving shelf whose `foodon_id` is that term OR a surviving ancestor of it. Run this audit *before* `fs.attach()` to catch projection bugs early; run it again after to spot any drift between the resolver here and the real one.

Read 5–10 of these by hand. The question to answer: **is each chunk genuinely about the shelf it lands on?** Common failure modes to look for:
- NEL drift: a chunk mentioning "milky way" got linked to `cow milk`.
- Over-broad attachment: a chunk explicitly about kale ends up on `vegetable` instead of `kale` because the more specific shelf got pruned.
- Off-topic match: text is about cardiology but linked to `fat` via "fat embolism".

In [ ]:
import random

# Index surviving shelves three ways:
#   shelf_by_foodon  : direct foodon_id → shelf  (chunk linked to this term)
#   shelf_by_seealso : collapsed foodon_id → shelf  (term collapsed into
#                                                    survivor; see_also)
# A chunk's mention of a collapsed term legitimately attaches to the
# survivor — same as a lifted ancestor walk, just recorded differently.
shelves = fs.graph_store.list_shelves()
shelf_by_foodon = {s.foodon_id: s for s in shelves if s.foodon_id is not None}
shelf_by_seealso = {fid: s for s in shelves for fid in s.see_also}

def project_chunk(foodon_ids):
    """Returns (attached_shelves, attachment_details) where details says how
    each foodon_id reached its shelf: 'direct' / 'collapsed' / 'lifted' / None.
    Mirrors what fs.attach() will compute once it lands."""
    attached = set()
    details = []  # list of (foodon_id, label, mode, shelf_label_or_none)
    for fid in foodon_ids:
        label = fs.ontology.id_to_label(fid) or fid
        # 1. direct: shelf with this foodon_id
        if fid in shelf_by_foodon:
            sh = shelf_by_foodon[fid]
            attached.add(sh.label)
            details.append((fid, label, 'direct', sh.label))
            continue
        # 2. collapsed: term was absorbed into a surviving descendant via see_also
        if fid in shelf_by_seealso:
            sh = shelf_by_seealso[fid]
            attached.add(sh.label)
            details.append((fid, label, 'collapsed', sh.label))
            continue
        # 3. lifted: deepest surviving ancestor (umbrella/blacklist drops left a gap)
        ancestors = fs.ontology.id_to_ancestors(fid) if fs.ontology else []
        best = None
        best_depth = -1
        for anc in ancestors:
            sh = shelf_by_foodon.get(anc) or shelf_by_seealso.get(anc)
            if sh is not None and sh.depth > best_depth:
                best = sh
                best_depth = sh.depth
        if best is not None:
            attached.add(best.label)
            details.append((fid, label, 'lifted', best.label))
        else:
            details.append((fid, label, 'orphan', None))
    return sorted(attached), details

# Collect chunks with ≥1 FOODON id.
candidates = []
for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        if chunk.foodon_ids:
            candidates.append(chunk)
        if len(candidates) >= 5000:  # cap upstream sampling cost
            break
    if len(candidates) >= 5000:
        break

# Corpus-wide counts: how many chunks with foodon_ids end up orphaned?
# (This is the systemic-bug check the audit asked for.)
n_orphaned = 0
n_collapsed_only = 0
n_lifted_only = 0
for chunk in candidates:
    _, details = project_chunk(chunk.foodon_ids)
    modes = {d[2] for d in details}
    if modes == {'orphan'}:
        n_orphaned += 1
    elif 'direct' not in modes and 'collapsed' in modes and 'lifted' not in modes:
        n_collapsed_only += 1
    elif 'direct' not in modes and 'lifted' in modes and 'collapsed' not in modes:
        n_lifted_only += 1

print(f'corpus-wide attachment audit on {len(candidates)} FOODON-linked chunks:')
print(f'  fully orphaned (no shelf):           {n_orphaned}  ← real bugs')
print(f'  reached shelf only via collapse:     {n_collapsed_only}')
print(f'  reached shelf only via lifting:      {n_lifted_only}')
if n_orphaned > 50:
    print(f'  ⚠ {n_orphaned} orphaned chunks is high; investigate systematic gap')
elif n_orphaned > 0:
    print(f'  ({n_orphaned} orphans likely from terms whose ancestry exits FOODON via non-food OBOs)')

# Sample 50 + show 10 with attachment annotations.
random.seed(0)
sample = random.sample(candidates, k=min(50, len(candidates)))
print(f'\n--- sample of {min(10, len(sample))} chunks (raise [:10] to [:50] for full audit) ---\n')
for i, chunk in enumerate(sample[:10]):
    attached, details = project_chunk(chunk.foodon_ids)
    # Group details by mode for compact display.
    direct_lines  = [f'{lbl!r}' for _, lbl, mode, _ in details if mode == 'direct']
    collapsed_lines = [f'{lbl!r} → {target}' for _, lbl, mode, target in details if mode == 'collapsed']
    lifted_lines = [f'{lbl!r} → {target}' for _, lbl, mode, target in details if mode == 'lifted']
    orphan_lines = [f'{lbl!r}' for _, lbl, mode, _ in details if mode == 'orphan']

    print(f'[{i+1}] {chunk.chunk_id}')
    print(f'    attached shelves:      {attached}')
    if direct_lines:
        print(f'    foodon → direct:       {direct_lines}')
    if collapsed_lines:
        print(f'    foodon → collapsed:    {collapsed_lines}')
    if lifted_lines:
        print(f'    foodon → lifted:       {lifted_lines}')
    if orphan_lines:
        print(f'    foodon → orphan:       {orphan_lines}  ← cannot reach any shelf')
    text = chunk.text.replace('\n', ' ')
    print(f'    text: {text[:220]}{"..." if len(text) > 220 else ""}')
    print()

# Drift candidates: (surface, ontology_id) pairs across the sample that
# repeat ≥ 3 times. These are candidates to add to
# `fs.config.layer_a.link_blocklist` if the audit shows they're
# semantically wrong. Surface forms are case-insensitive in the blocklist.
from collections import Counter
drift_counter: Counter = Counter()
for chunk in sample:
    for link in chunk.entity_links:
        if link.ontology_id.startswith('FOODON:'):
            drift_counter[(link.mention.text.lower(), link.ontology_id)] += 1

frequent = [(pair, n) for pair, n in drift_counter.most_common() if n >= 3]
if frequent:
    print(f'\n--- (surface, ontology_id) pairs appearing ≥3 times across {len(sample)} sampled chunks ---')
    print('add to link_blocklist if semantically wrong:')
    for (surface, oid), n in frequent[:15]:
        label = fs.ontology.id_to_label(oid) or oid
        print(f'  {n:>3d}× ({surface!r:>30}, {oid:25s}) → label={label!r}')
else:
    print('\nno (surface, ontology_id) pair repeats ≥3 times — no obvious drift in this sample.')


### 5f. Sanity gate — 20 chunks with non-FOODON links only

These are the chunks that reached the foods facet only via the synthetic root (no FOODON id, but ≥1 entity_link of some other prefix). The question: **does the text contain a food term that should have linked but didn't?** That's a NEL coverage gap — the upstream linker missed a food mention.

If 5+ of 20 sampled chunks have an obvious missed food term, re-annotation with GLiNER is the right next step. If most are nutrition-y-but-not-food (papers about "macronutrient distribution", clinical population studies, etc.), the synthetic-root lift is expected and harmless.

In [ ]:
candidates_nonfoodon = []
for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        if not chunk.foodon_ids and chunk.entity_links:
            candidates_nonfoodon.append(chunk)
        if len(candidates_nonfoodon) >= 5000:
            break
    if len(candidates_nonfoodon) >= 5000:
        break

random.seed(1)
sample_nf = random.sample(candidates_nonfoodon, k=min(20, len(candidates_nonfoodon)))
print(f'sampled {len(sample_nf)} of {len(candidates_nonfoodon)} non-FOODON-only chunks scanned\n')

for i, chunk in enumerate(sample_nf):
    surface_forms = [link.mention.text for link in chunk.entity_links[:5]]
    prefixes = sorted({link.ontology_id.split(':', 1)[0] for link in chunk.entity_links})
    print(f'[{i+1}] {chunk.chunk_id}')
    print(f'    linked mentions ({len(chunk.entity_links)}): {surface_forms}{" ..." if len(chunk.entity_links) > 5 else ""}')
    print(f'    prefixes seen: {prefixes}')
    text = chunk.text.replace("\n", " ")
    print(f'    text: {text[:260]}{"..." if len(text) > 260 else ""}')
    print()

## 6. Attach chunks to shelves

`fs.attach()` is the bridge phase between Layer A projection and everything downstream. It walks every chunk, resolves each FOODON id to a surviving shelf via the same direct → collapsed → lifted cascade the §5d audit previews, then writes:

- **Neo4j**: `(:Chunk)-[:ATTACHED_TO {lifted_from: [foodon_id,...]}]->(:Shelf)` edges. `lifted_from` is empty for direct attachments (chunk linked the shelf's own term); non-empty when the chunk reached the shelf via collapse or by lifting through ancestors.
- **Elastic**: each chunk's `shelf_ids` keyword-array gets the union of its resolved shelves, so retrieval can filter `terms { shelf_ids: [...] }` without crossing the graph.

Attachment is **nearest surviving ancestor only** (one shelf per FOODON id, the deepest match), facet-aware (`route_link_to_facet` keeps a `medical condition` mention out of the foods walk), and idempotent (re-running with the same projection produces the same edges and the same `lifted_from` payloads). Orphan FOODON ids whose ancestry can't reach any real shelf fall through to the synthetic facet root (`facet:foods`, etc.).

In [6]:
meta = fs.attach()
print(meta)

# Per-shelf attachment counts straight from Neo4j. Order matches the §5d
# audit's "top 10 foods shelves by chunk_count" so any discrepancy between
# projection-time `chunk_count` and post-attach edge count flags drift —
# typically a chunk whose deepest surviving ancestor differs from the term
# that drove support during projection (e.g. when collapse rewrites the
# parent chain after support was already counted).
from collections import Counter

attach_counts: Counter = Counter()
lifted_edges = 0
direct_edges = 0
with fs.graph_store._driver.session() as session:  # type: ignore[attr-defined]
    result = session.run(
        """
        MATCH (c:Chunk)-[r:ATTACHED_TO]->(s:Shelf)
        WHERE s.facet = 'foods'
        RETURN s.shelf_id AS sid, s.label AS label,
               count(*) AS n,
               sum(CASE WHEN size(r.lifted_from) = 0 THEN 1 ELSE 0 END) AS direct,
               sum(CASE WHEN size(r.lifted_from) > 0 THEN 1 ELSE 0 END) AS lifted
        ORDER BY n DESC
        LIMIT 12
        """
    )
    print(f'\ntop foods shelves by attached chunks (graph edges):')
    for row in result:
        attach_counts[row['sid']] = row['n']
        direct_edges += row['direct']
        lifted_edges += row['lifted']
        print(f'  {row["label"][:40]:40s}  {row["n"]:>5d}  '
              f'(direct={row["direct"]}, via lifted_from={row["lifted"]})')

print(f'\nedge breakdown in top 12: direct={direct_edges}, lifted/collapsed={lifted_edges}')

# Pick one chunk and show what it landed on — concrete confirmation that
# the in-Elastic shelf_ids denorm matches the graph edges, and that
# lifted_from payloads survive the round-trip.
sample_chunk = next(
    (c for c in fs.chunk_store.scan() if c.shelf_ids), None
)
if sample_chunk is not None:
    print(f'\nsample chunk {sample_chunk.chunk_id!r}:')
    print(f'  foodon_ids on chunk:    {sample_chunk.foodon_ids[:6]}')
    print(f'  shelf_ids (Elastic):    {sample_chunk.shelf_ids}')
    with fs.graph_store._driver.session() as session:  # type: ignore[attr-defined]
        result = session.run(
            """
            MATCH (c:Chunk {chunk_id: $cid})-[r:ATTACHED_TO]->(s:Shelf)
            RETURN s.label AS label, s.shelf_id AS sid, r.lifted_from AS lifted_from
            """,
            cid=sample_chunk.chunk_id,
        )
        print(f'  Neo4j edges:')
        for row in result:
            mode = 'direct' if not row['lifted_from'] else f'lifted_from={row["lifted_from"]}'
            print(f'    -> {row["label"]!r:40s}  ({mode})')

2026-05-26T18:57:46.684612Z [info     ] neo4j.attachments_cleared      url=bolt://localhost:7687
POST http://localhost:9200/foodscholar_chunks/_update_by_query?refresh=true&wait_for_completion=true [status:200 duration:17.479s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.067s]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.455s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.068s]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.446s]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.450s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.080s]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.422s]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duration:0.385s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.080s]
PUT http://localhost:9200/_bulk?refresh=false [status:200 duratio

artifact_id='attach-98bdb608e354' phase='attach' config_hash='3e95e5b0e58fdda1' upstream_artifact_ids=[] record_count=23827 schema_version='v0.1' created_at=datetime.datetime(2026, 5, 26, 18, 59, 8, 334394, tzinfo=datetime.timezone.utc)

top foods shelves by attached chunks (graph edges):


POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.056s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.045s]


  Foods                                      4276  (direct=0, via lifted_from=4276)
  fat                                         868  (direct=585, via lifted_from=283)
  fruit produce                               633  (direct=633, via lifted_from=0)
  vegetable                                   542  (direct=542, via lifted_from=0)
  cow milk                                    522  (direct=522, via lifted_from=0)
  red meat (raw)                              478  (direct=69, via lifted_from=409)
  fish species                                471  (direct=396, via lifted_from=75)
  poultry                                     380  (direct=366, via lifted_from=14)
  dietary supplement                          332  (direct=265, via lifted_from=67)
  sugar                                       300  (direct=293, via lifted_from=7)
  egg or egg component                        272  (direct=270, via lifted_from=2)
  vitamin preparation                         258  (direct=258, via lifted_from=

POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.046s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.041s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.040s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.046s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.056s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.048s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.066s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.060s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.047s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.041s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.042s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.052s]
POST


sample chunk '0001964d-a3ef-4fd0-a641-15d532348577':
  foodon_ids on chunk:    ['FOODON:03317626']
  shelf_ids (Elastic):    ['foodon:FOODON:03301955']
  Neo4j edges:
    -> 'bacon (raw)'                             (lifted_from=['FOODON:03317626'])


## 6b. Audit + quality report

Two checks, two different questions:

- **`fs.audit()`** — *is the graph correctly built?* Five invariants across the chunk store, entity store, and graph store. Critical failures mean Layer B will build on a broken Layer A. Use this to gate downstream work.
- **`fs.quality_report(facet='foods')`** — *is the graph good?* A domain-expert-facing report with five sections (top shelves, hierarchy walkthrough, suspicious shelves, canonical vocabulary check, random chunk sample) formatted as Markdown for in-notebook review.

Audit answers yes/no with hard thresholds; quality is a conversation a nutritionist reads top-to-bottom in ~30 minutes. The quality report is intentionally NOT a single score — quality is multidimensional, and collapsing it would hide the failures that matter.

In [7]:
# Audit first — invariants. If this fails, the rest is on shaky ground.
audit = fs.audit()
print(audit)
print()
if audit.critical_failures:
    print(f'⚠ {len(audit.critical_failures)} critical invariant(s) failing. '
          'Inspect audit.critical_failures[i].sample for examples.')

# Then quality. Long Markdown — render via IPython for in-notebook readability.
from IPython.display import Markdown

quality = fs.quality_report(facet='foods', top_n=15, sample_size=20, seed=0)
Markdown(str(quality))

POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.055s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.049s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.047s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.048s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.052s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.054s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.042s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.043s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.049s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.043s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.056s]
POST http://localhost:9200/foodscholar_chunks/_search [status:200 duration:0.061s]
POST

Audit report  (config_hash=4982fd8cd2e37150, generated_at=2026-05-26T19:01:01.541330+00:00)

Inventory:
  chunks_total                    13344
  shelves_allergies               1
  shelves_dietary_patterns        1
  shelves_foods                   227
  shelves_health                  1
  shelves_nutrients               1
  shelves_sustainability          1
  shelves_total                   232

B. Coverage
  [PASS INFO] overall chunk attachment rate — metric=0.4714
  [PASS CRITICAL] FOODON-linked chunks reaching a shelf — metric=1.0, threshold=0.95
C. Cross-store consistency
  [PASS CRITICAL] foodon_ids (Elastic) ↔ MENTIONS (Neo4j) exact match — metric=1.0, threshold=0.95
D. Attach integrity
  [PASS CRITICAL] shelf_ids (Elastic) ↔ ATTACHED_TO (Neo4j) parity — metric=1.0, threshold=0.99
  [PASS WARNING] projection support_direct ↔ actual direct edges — metric=0, threshold=11
E. Structural sanity
  [PASS CRITICAL] dangling parent references — metric=0, threshold=0
  [PASS CRITICAL] cy

POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.006s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.004s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.005s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.004s]
POST http://localhost:9200/foodscholar_chunks/_mget [status:200 duration:0.003s]
POST http://localhost:9200/f

# Layer A quality report — facet='foods'

_config_hash 4982fd8cd2e37150, generated 2026-05-26T19:01:02.793585+00:00_
_227 shelves in this facet_

## 1. Top shelves at a glance

### 'Foods' (depth 0, 5135 chunks, direct 0 / lifted 5135)
  - Here are some ideas for adding interest to the taste of plain water without lowcalorie sweeteners, colors, or flavors or too much added sugar.
  - - . . . what spa cuisine is? Although the term isn't regulated, spa cuisine often refers to health-positioned food preparation, perhaps promoted in resorts or health club cafés. On a menu that offers 'spa cuisine' you m…
  - Treatment for lead poisoning includes removing the child from the source of contamination and extracting lead from the body. Extraction may involve chelation therapy, which binds with lead so it can be excreted in urine…

### 'fruit produce' (depth 1, 633 chunks, direct 633 / lifted 0)
  - | To Determine: | Ask: | |----------------------------------|--------------------------------------------------------------------------------------------------| | D isease | ■ Do you have an illness or condition that ch…
  - Excessive weight gain has become an epidemic. According to the National Institutes of Health, over two-thirds of American adults are overweight, and one in three is obese. Obesity in particular puts people at risk for a…
  - How Irradiation Works Irradiation exposes foods to controlled doses of gamma rays from the radioactive element cobalt 60. As the rays pass through living cells, they disrupt DNA, proteins, and other internal structures,…

### 'fat' (depth 1, 596 chunks, direct 585 / lifted 11)
  - In the Small Intestine Some of the sweet sugars in the banana require so little digesting that they begin to cross the linings of the small intestine immediately on contact. Nearby, the liver donates bile through a duct…
  - As for saturated and trans fats, they pose the same heart disease risk for athletes as they do for other people. Physical activity reduces the risk of cardiovascular disease, but athletes still suffer heart attacks and …
  - - /H18567 Emphasizes fruits, vegetables, whole grains, and fat-free or low-fat milk and milk products - /H18567 Includes lean meats, poultry, fish, beans, eggs, and nuts - /H18567 Is low in saturated fats, trans fats, c…

### 'legume' (depth 1, 546 chunks, direct 233 / lifted 313)
  - To prevent iron deficiency, children's foods must deliver 7 to 10 milligrams of iron per day. To achieve this goal, snacks and meals should include iron-rich foods. Although milk is an important source of dietary calciu…
  - Yet, it' s important to choose your 'carbs' wisely! The Dietary Guidelines advise: choose fiber-rich fruits, vegetables, and whole grains often because they're nutrient-rich! Enjoy mostly whole fruit, which has more fib…
  - - ■ Folate and vitamin B 12 play key roles in cell replication and are needed in large amounts during pregnancy. - ■ Folate plays an important role in preventing neural tube defects. Choline Although not defined as a vi…

### 'vegetable' (depth 1, 542 chunks, direct 542 / lifted 0)
  - Yet, it' s important to choose your 'carbs' wisely! The Dietary Guidelines advise: choose fiber-rich fruits, vegetables, and whole grains often because they're nutrient-rich! Enjoy mostly whole fruit, which has more fib…
  - | To Determine: | Ask: | |----------------------------------|--------------------------------------------------------------------------------------------------| | D isease | ■ Do you have an illness or condition that ch…
  - - /H18567 Season with herbs, spices, herbed vinegars, herb rubs, and fruit juices instead of salt. 'A Pinch of Flavor: How to Cook with Herbs and Spices' in chapter 13 offers many ways to use herbs and spices. Or prepar…

### 'cow milk' (depth 1, 522 chunks, direct 522 / lifted 0)
  - To prevent iron deficiency, children's foods must deliver 7 to 10 milligrams of iron per day. To achieve this goal, snacks and meals should include iron-rich foods. Although milk is an important source of dietary calciu…
  - Have you ever heard parents say that their child is allergic to milk, then remark that he or she has no adverse reactions to chocolate milk? Or maybe you avoid a particular food yourself, believing you have an allergy t…
  - | FOOD | SERVING | VITAMIND | |----------------------------|-------------------|---------------| | Milk with added vitaminD | 200 mLglass | 4 μg | | Cereal with added vitaminD | 30-40 g | 1.5 to 2.9 μg | | Yogurt with a…

### 'fish species' (depth 1, 494 chunks, direct 396 / lifted 98)
  - To prevent iron deficiency, children's foods must deliver 7 to 10 milligrams of iron per day. To achieve this goal, snacks and meals should include iron-rich foods. Although milk is an important source of dietary calciu…
  - How much protein do you need for sports? Base the amount on your body weight, not your energy need. /H18567 For most recreational exercisers, 0.5 to 0.75 gram of protein per pound of body weight is enough. (The upper en…
  - As for saturated and trans fats, they pose the same heart disease risk for athletes as they do for other people. Physical activity reduces the risk of cardiovascular disease, but athletes still suffer heart attacks and …

### 'dietary supplement' (depth 1, 435 chunks, direct 265 / lifted 170)
  - - /H18567 Keep supplements in a cool, dry place-preferably away from the stove and not in the bathroom. Heat and moisture affect their quality and effectiveness. Keep them in their original containers (label still on). …
  - Which supplement to choose? The first step is to remain aware that sales of vitamin supplements often approach the | Table C7-3 | Table C7-3 | Table C7-3 | Table C7-3 |
  - | Sucralose (Splenda) | Sucrose with CI atoms instead of OH groups | Not digested or absorbed | 600 | 0 | 5 mg/kg body weight (6 cans diet soda) | Baked goods, carbon- ated beverages, chewing gum, coffee and tea, dairy …

### 'poultry' (depth 1, 380 chunks, direct 366 / lifted 14)
  - To prevent iron deficiency, children's foods must deliver 7 to 10 milligrams of iron per day. To achieve this goal, snacks and meals should include iron-rich foods. Although milk is an important source of dietary calciu…
  - Most sodium in the typical American diet comes from processed and prepared foods. Manufacturers add salt to foods to improve texture and flavor, and also as a preservative. The amount of salt in similar food products va…
  - How much protein do you need for sports? Base the amount on your body weight, not your energy need. /H18567 For most recreational exercisers, 0.5 to 0.75 gram of protein per pound of body weight is enough. (The upper en…

### 'snack food' (depth 1, 345 chunks, direct 127 / lifted 218)
  - - get rewarding or punishing a child with food. In that way you won't reinforce an emotional link to eating. - /H18567 Stock your kitchen with lower-calorie snack choices such as raw vegetables, fruit, milk, or vanilla …
  - - /H18567 More indulgence. There' s a flip side to the growing array of foods marketed for health. There are now more indulgence foods: richer, higher-fat frozen desserts, bigger portions, and more high-calorie snack fo…
  - Choose grain products by the company they keep! Make at least half your choices whole grain for their fiber and other benefits. Go easy on those made with more fat, cholesterol, and added sugars, such as croissants, pas…

### 'confectionery food' (depth 1, 314 chunks, direct 36 / lifted 278)
  - Many processed foods like ready meals, cakes and confectionery contain high levels of added sugars and can be high in calories. Some soft drinks contain mostly sugar and provide very little nourishment. Cutting down on …
  - Choose grain products by the company they keep! Make at least half your choices whole grain for their fiber and other benefits. Go easy on those made with more fat, cholesterol, and added sugars, such as croissants, pas…
  - such as biscuits, snacks, savoury snacks and confectionery. These foods are not needed for healthy eating and are high in calories, sugar and salt.

### 'processed meat food product' (depth 1, 313 chunks, direct 143 / lifted 170)
  - list a list, established by the FDA, of food additives long in use and believed to be safe. margin of safety in reference to food additives, a zone between the concentration normally used and that at which a hazard exis…
  - Lactose usually comes from foods containing milk or milk solids. The chart 'How Much Lactose?' on page 524 suggests how much. Prepared foods, even those labeled 'nondairy,' may contain lactose. If you're very lactose-in…
  - - /H18567 If you receive a food gift, open it right away. Some mail-order foods, such as dry-cured ham and hard salami, don't require refrigeration. Other meats (including most hams), poultry, fish, and other perishable…

### 'sugar' (depth 1, 302 chunks, direct 293 / lifted 9)
  - The cavity-producing process starts when bacteria in your mouth mix with carbohydrates-both sugars and The glycemic index of a single food isn't reliable for helping most healthy consumers make food choices. Why? First,…
  - Many processed foods like ready meals, cakes and confectionery contain high levels of added sugars and can be high in calories. Some soft drinks contain mostly sugar and provide very little nourishment. Cutting down on …
  - - NOT every day - Maximum once or twice a week These foods are high in calories, fat, sugar and salt, but low in fibre, vitamins and minerals.

### 'egg or egg component' (depth 1, 272 chunks, direct 270 / lifted 2)
  - To prevent iron deficiency, children's foods must deliver 7 to 10 milligrams of iron per day. To achieve this goal, snacks and meals should include iron-rich foods. Although milk is an important source of dietary calciu…
  - The meal consists of one medium-sized potato, half a large cut of meat, half a smaller cut, 2 to 3 slices of lean meat (such as ham or chicken/turkey breast), 4 slices of ham, 1 whole egg, 4 tablespoons of cooked beans,…
  - | Calcium | Fortified cereals | Dark-green leafy vegeta- bles (bok choy, broccoli, collard greens, kale, mustard greens, turnip greens, watercress) | Fortified juices, figs | Fortified soy products, nuts (almonds), seed…

### 'vitamin preparation' (depth 1, 258 chunks, direct 258 / lifted 0)
  - The energy a person consumes in a day's meals comes from these three energy-yielding nutrients; alcohol, if consumed, also contributes energy at a rate of about 7 calories per gram (see note). | Energy Nutrient | Energy…
  - | Characteristics of the Fat-Soluble and Water-Soluble Vitamins | Characteristics of the Fat-Soluble and Water-Soluble Vitamins | Characteristics of the Fat-Soluble and Water-Soluble Vitamins | | Although each vitamin h…
  - | FOOD | SERVING | VITAMIND | |----------------------------|-------------------|---------------| | Milk with added vitaminD | 200 mLglass | 4 μg | | Cereal with added vitaminD | 30-40 g | 1.5 to 2.9 μg | | Yogurt with a…

## 2. Hierarchy walkthrough

### Foods
  parents: (root)
  descendants: 38910 - french fries from cut potato (efsa foodex2), almond, animal egg, animal meat (ground, extra lean), animal-derived food, antioxidant (ec), apple, apricot food product

### fruit produce
  parents: Foods

### fat
  parents: Foods

### legume
  parents: Foods
  descendants: kidney bean (raw), lentil, pea, peanut

### vegetable
  parents: Foods

## 3. Suspicious shelves

- **'38910 - french fries from cut potato (efsa foodex2)'** — label has EFSA-style numeric code prefix (label='38910 - french fries from cut potato (efsa foodex2)')
- **'cream (raw)'** — label carries a processing-state qualifier (label='cream (raw)')
- **'food (canned)'** — label carries a processing-state qualifier (label='food (canned)')
- **'produce (raw)'** — label carries a processing-state qualifier (label='produce (raw)')
- **'red meat (raw)'** — label carries a processing-state qualifier (label='red meat (raw)')
- **'seed (raw)'** — label carries a processing-state qualifier (label='seed (raw)')
- **'tuna (raw)'** — label carries a processing-state qualifier (label='tuna (raw)')
- **'bacon (raw)'** — label carries a processing-state qualifier (label='bacon (raw)')
- **'kidney bean (raw)'** — label carries a processing-state qualifier (label='kidney bean (raw)')
- **'tofu (raw)'** — label carries a processing-state qualifier (label='tofu (raw)')

## 4. Canonical vocabulary check

- ✓ 'olive oil' → shelf 'olive oil' (53 chunks)
- ~ 'kale' → exists in FoodOn as FOODON:03000235 but no surviving shelf
- ~ 'broccoli' → exists in FoodOn as FOODON:00003405 but no surviving shelf
- ~ 'quinoa' → exists in FoodOn as FOODON:03310193 but no surviving shelf
- ✓ 'salmon' → shelf 'salmon' (67 chunks)
- ✓ 'walnut' → shelf 'walnut' (28 chunks)
- ~ 'yogurt' → exists in FoodOn as FOODON:00003193 but no surviving shelf
- ✓ 'whole grain' → shelf 'whole grain' (215 chunks)
- ✓ 'legume' → shelf 'legume' (546 chunks)
- ✓ 'blueberry' → shelf 'blueberry' (32 chunks)
- ~ 'omega 3' → exists in FoodOn as FOODON:03510245 but no surviving shelf
- ~ 'vitamin D' → exists in FoodOn as FOODON:00005480 but no surviving shelf
- ~ 'fiber' → exists in FoodOn as FOODON:03401328 but no surviving shelf
- ~ 'iron' → exists in FoodOn as FOODON:00003832 but no surviving shelf
- ~ 'calcium' → exists in FoodOn as FOODON:03307487 but no surviving shelf
- ~ 'vitamin B12' → exists in FoodOn as FOODON:03544482 but no surviving shelf
- ✓ 'antioxidant' → shelf 'antioxidant (ec)' (39 chunks)
- ~ 'polyphenol' → exists in FoodOn as FOODON:03544508 but no surviving shelf
- ✗ 'mediterranean diet' — not found
- ✗ 'DASH diet' — not found
- ✗ 'vegetarian diet' — not found
- ✗ 'ketogenic diet' — not found
- ✗ 'cardiovascular disease' — not found
- ✗ 'diabetes' — not found
- ✗ 'obesity' — not found
- ✗ 'hypertension' — not found
- ✗ 'inflammation' — not found
- ✗ 'insulin resistance' — not found
- ✗ 'metabolic syndrome' — not found

## 5. Random chunk sample (for hand audit)

_Read each chunk and ask: does it really belong on the listed shelves? Mark yes/no manually._

### [1] 7e848a64-485e-42a9-a5b3-0e7d9e284847 (textbook)
  shelves: food supplement form
  text: Loss of appetite and impaired nutrient absorption interfere with attempts to provide nourishment to a malnourished child. Even with hospital care, many children do not recover from SAM-their malnutrition proves fatal. 14 For a chance to restore metabolic balance and to resume ph…

### [2] fcb20b35-a5ad-43aa-a526-75e53ba1a7aa (textbook)
  shelves: Foods, apricot food product, berry food product, carrot food product, spinach food product, fruit (fresh), sweet potato, rice, cow milk, raisin, fruit (dried), oatmeal (cooked), breakfast product made from cereals
  text: - /H18567 Add ginger to a fruit glaze. Blend frozen raspberries with a pinch of ginger and a small amount of fruit juice concentrate or sweetener. Toss the glaze with fresh berries or sliced fruit. - /H18567 Add a 'sweet' spice of your choice to dry coffee grounds before brewing…

### [3] 8aa1ea25-5b86-41a0-81e4-42fef62f69a6 (textbook)
  shelves: Foods
  text: - ■ Natural foods contain natural toxins that can be hazardous under some conditions. - ■ To avoid harm from toxins, choose a variety of foods and eat them in moderation.

### [4] 0c9c107b-5ca7-44e4-88e9-46207fa10ef3 (textbook)
  shelves: Foods
  text: Catabolism of food molecules begins when food enters the mouth, as the enzyme salivary amylase initiates the breakdown of the starch in foods. The entire process of digestion converts the large polymers in food to monomers that can be absorbed. Starches are broken down to monosa…

### [5] 53a1b5a4-6f4d-4893-999e-14cc11056c9f (textbook)
  shelves: Foods, malnutrition supplement food product
  text: In contrast to malnourished adults, young impoverished and malnourished children often exhibit specific, more readily identifiable conditions. The form malnutrition takes in a hungry child depends partly on the nature of the food shortage that caused it. The most perilous condit…

### [6] a89f8867-7134-42ca-afa8-0ab31bea8975 (textbook)
  shelves: fat
  text: Unlike other body cells that can store fat in limited supplies, fat cells are specialized for fat storage and are able to expand almost indefinitely in size. An overabundance of adipose tissue can result in undue stress on the body and can be detrimental to your health. A seriou…

### [7] 9f649ca3-ba78-4511-8628-8da780df25ff (textbook)
  shelves: Foods
  text: In the end, people choose foods, not individual nutrients such as saturated fat, and their choices, repeated over time, form habitual dietary patterns that greatly affect their health. For this reason, the Dietary Guidelines committee adopts a dietary pattern approach to help Am…

### [8] 853da040-10a3-4604-9021-b59d119d3469 (guide)
  shelves: vegetable, rice, whole wheat bread, pasta, potato, salad, fruit produce
  text: | Wholemeal cereals and breads, potatoes, pasta and rice | Moderately active:* 4 to 5 servings per day Inactive: 3 to 4 servings per day (*very active adult women will need more) | | Vegetables, salad and fruit 5 to 7 | servings per day |

### [9] 62596b8c-55f6-44e4-9c1c-3ebf1f282323 (textbook)
  shelves: Foods, fat, serving size
  text: One classic cinnamon roll can have 5 grams of trans fat, which is quite high for a single snack. Many packaged foods often have their nutrient contents listed for a very small serving size-much smaller than what people normally consume-which can easily lead you to eat many 'serv…

### [10] 9ca12b77-ee60-4b1f-b7bf-b1b85c3f1f24 (textbook)
  shelves: thiamin mononitrate
  text: Vitamin D is the most potentially toxic among the vitamins. Vitamin D intoxication raises the concentration of blood calcium by withdrawing bone calcium, which can then collect in the soft tissues and damage them. With chronic high vitamin D intakes, kidney and heart function de…

### [11] 740fa3af-c555-4ac7-a814-22387524aee8 (textbook)
  shelves: Foods, coffee beverage, soft drink
  text: 'Excessive' caffeine intake is individual. Caffeine sensitivity depends on the amount and frequency of caffeine intake, body weight, physical condition, and overall anxiety level, among other factors. Tolerance to caffeine develops over time. A regular coffee drinker may not not…

### [12] c01e946d-9098-46be-9e20-8f356e4852d9 (guide)
  shelves: fat
  text: 3 Trans fats are also known as hydrogenated (or partially hydrogenated) fats or hydrogenated (or partially hydrogenated) vegetable oils

### [13] 472d5dec-63e8-4b00-9408-89a88c1f58de (textbook)
  shelves: Foods, cow milk
  text: Vitamin D promotes calcium absorption. If you drink vitamin D-fortified milk, you probably consume enough to protect against bone disease. But if your calcium comes from other sources, get a little sunshine. Your body makes vitamin D when ultraviolet light touches your skin. If …

### [14] a5b3da55-dd5a-44d3-ae46-d994897ea8ae (textbook)
  shelves: Foods, broccoli food product, hummus, yellow onion, pepper, mango, vegetable, tomato, peach, celery plant
  text: - /H18567 Eat vegetables and fruits raw. Or cook many vegetables, such as asparagus, green beans, broccoli, and snow peas, quickly-just until tender-crisp. Some nutrients, such as B vitamins and vitamin C, are destroyed easily by heat. The shorter the cooking time, the more nutr…

### [15] 2c7f70bf-7c59-41eb-84a2-54458d4c39a5 (textbook)
  shelves: Foods, prepared seafood product, fish species, shrimp
  text: Be aware of differences in nutritional content when you make substitutions. Squid, shrimp, and lobsters, for example, have more cholesterol than clams, crabs, mussels, and scallops do. Three ounces of boiled shrimp supply about 165 milligrams of cholesterol; the same amount of s…

### [16] 5b2f61f0-d4ae-41fe-b5b3-ea25a84c101d (textbook)
  shelves: Foods, meat product analog, olive food product, hummus, popcorn food product, fat, avocado, nuts (mixed), pasta, potato
  text: - Grill, roast, broil, boil, bake, stir-fry, microwave, or poach foods. Don't fry in shortening, lard, or butter. Try pan frying in a few teaspoons of olive or vegetable oil instead of deep-fat frying. - Reduce or eliminate food 'add-ons' such as buttery, cheesy, or creamy sauce…

### [17] 2ca8171e-665c-4460-9fda-c4db33ea8a6f (textbook)
  shelves: vegetable, whole grain, nuts (mixed), legume, fruit produce, fish species
  text: A main objective of a defensive diet is to lower blood triglycerides and cholesterolthat is, to reduce blood VLDL and LDL. And just as a diet high in saturated and trans fats raises LDL, a diet low in those lipids lowers them. Where people in the world consume diets high in satu…

### [18] fbdcde7e-8ff7-4b2d-b119-c87b4b9fc060 (textbook)
  shelves: dietary supplement
  text: By far, most people can meet their nutrient needs from their diets alone. * Reference notes are in Appendix F. Indisputably, however, the people listed in Table C7-1 need supplements. For them, nutrient supplements can prevent or reverse illnesses. Because supplements are not ri…

### [19] 1d9d358a-fce2-4a49-97c1-7cf06fda69f4 (textbook)
  shelves: fruit produce, antioxidant (ec)
  text: Where should your antioxidant vitamins come from? An eating style with plenty of fruits and vegetables is undisputed as the wisest approach to good health.

### [20] caaac242-2e1c-4ce2-9403-b04c5aaae015 (textbook)
  shelves: Foods, processed meat food product, fruit (fresh), poultry, shrimp
  text: The FDA has evaluated the safety of irradiated food for more than 30 years and has found the process to be safe. The World Health Organization (WHO), the Centers for Disease Control and Prevention (CDC) and the U.S. Department of Agriculture (USDA) have also endorsed the safety …


### 6c. Layer A connectivity profile

The audit answers *is the graph correctly wired?* with yes/no invariants; this cell answers
*what does the wiring **look like**?* — the quantitative shape of Layer A's connectivity, the
diagnostics you'd skim before trusting downstream Layer B clustering.

Two graphs make up Layer A's connectivity and we profile both:

- **Shelf tree** (`(:Shelf)-[parent]->(:Shelf)`) — the navigation backbone. We read it
  straight off `parent_shelf_id` so this works on *any* backend (Neo4j or in-memory). Panels
  cover **branching factor** (children per shelf), **depth profile per facet**, and the
  **support-weight CCDF** (how chunk_count concentrates across shelves).
- **Chunk→shelf bipartite layer** (`(:Chunk)-[:ATTACHED_TO]->(:Shelf)`) — the actual
  attachment fan-out. Pulled from Neo4j edges when the graph store exposes a driver; the panel
  is skipped with a note on backends that don't (the projection-side `chunk_count` already
  covers the same ground in §5).

Read it as: a healthy backbone is a *tree* (mean branching 2–5, few orphan roots), support is
**heavy-tailed but not single-peaked** (a handful of broad shelves + a long body of specific
ones), and the synthetic facet root absorbs a *minority* of attachments (a majority means NEL
coverage is thin — see the §5d audit).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

FACET = 'foods'  # profile one facet; flip to None to include every facet.

shelves = fs.graph_store.list_shelves()
if FACET is not None:
    shelves = [s for s in shelves if s.facet == FACET]
by_id = {s.shelf_id: s for s in shelves}

# --- Shelf-tree structure (backend-agnostic: read off parent_shelf_id) ------
children: dict[str, list] = defaultdict(list)
roots = []
for s in shelves:
    pid = s.parent_shelf_id
    if pid is not None and pid in by_id:
        children[pid].append(s)
    else:
        roots.append(s)  # depth-0 root, or parent pruned out of this facet

# Branching factor across *internal* shelves (those with ≥1 child). Leaves
# (zero children) are reported separately so they don't swamp the histogram.
branch_counts = [len(children[s.shelf_id]) for s in shelves]
n_leaves = sum(1 for b in branch_counts if b == 0)
internal_branch = [b for b in branch_counts if b > 0]
mean_branch = float(np.mean(internal_branch)) if internal_branch else 0.0

# Depth profile per facet (or single facet) for the connectivity backbone.
depth_by_facet: dict[str, Counter] = defaultdict(Counter)
for s in shelves:
    depth_by_facet[s.facet][s.depth] += 1

print(f'Layer A backbone ({FACET or "all facets"}):')
print(f'  shelves:            {len(shelves)}')
print(f'  roots (orphans):    {len(roots)}  ({", ".join(s.label[:24] for s in roots[:4])}'
      f'{" …" if len(roots) > 4 else ""})')
print(f'  leaves:             {n_leaves}')
print(f'  internal shelves:   {len(internal_branch)}  (mean branching {mean_branch:.2f})')
print(f'  max depth:          {max((s.depth for s in shelves), default=0)}')

# --- Chunk→shelf attachment degrees from real ATTACHED_TO edges (Neo4j) -----
# Falls back to projection-time chunk_count if the backend has no driver.
attach_degree: dict[str, int] = {}
synthetic_root_share = None
driver = getattr(fs.graph_store, '_driver', None)
if driver is not None:
    facet_clause = 'WHERE s.facet = $facet' if FACET is not None else ''
    with driver.session() as session:
        rows = session.run(
            f'''
            MATCH (c:Chunk)-[:ATTACHED_TO]->(s:Shelf)
            {facet_clause}
            RETURN s.shelf_id AS sid, count(c) AS n
            ''',
            facet=FACET,
        )
        attach_degree = {r['sid']: r['n'] for r in rows}
    total_edges = sum(attach_degree.values())
    # Synthetic facet root id is `facet:<name>` per fs.attach()'s orphan fallback.
    synth_id = f'facet:{FACET}' if FACET is not None else None
    if synth_id and total_edges:
        synthetic_root_share = attach_degree.get(synth_id, 0) / total_edges
    print(f'\nattachment edges (graph): {total_edges}')
    if synthetic_root_share is not None:
        flag = '  ⚠ majority — NEL coverage likely thin' if synthetic_root_share > 0.5 else ''
        print(f'  synthetic root share:   {synthetic_root_share:.1%}{flag}')
else:
    attach_degree = {s.shelf_id: s.chunk_count for s in shelves}
    print('\n(no Neo4j driver — using projection-time chunk_count for the fan-out panel)')

# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Panel 1 — branching-factor distribution (children per internal shelf).
ax = axes[0, 0]
if internal_branch:
    bins = np.arange(0.5, max(internal_branch) + 1.5, 1)
    ax.hist(internal_branch, bins=bins, color='steelblue', edgecolor='white')
    ax.axvline(mean_branch, color='crimson', linestyle='--',
               label=f'mean = {mean_branch:.2f}')
    ax.legend()
ax.set_title(f'Branching factor\n({len(internal_branch)} internal shelves, '
             f'{n_leaves} leaves not shown)')
ax.set_xlabel('children per shelf')
ax.set_ylabel('# shelves')
ax.grid(True, alpha=0.3)

# Panel 2 — depth profile (stacked by facet if FACET is None, else single bar set).
ax = axes[0, 1]
all_depths = sorted({d for c in depth_by_facet.values() for d in c})
facets_sorted = sorted(depth_by_facet)
bottoms = np.zeros(len(all_depths))
for facet in facets_sorted:
    heights = np.array([depth_by_facet[facet].get(d, 0) for d in all_depths])
    ax.bar([str(d) for d in all_depths], heights, bottom=bottoms, label=facet)
    bottoms += heights
ax.set_title('Shelf count by projection depth')
ax.set_xlabel('depth')
ax.set_ylabel('# shelves')
if len(facets_sorted) > 1:
    ax.legend(title='facet', ncol=2)
ax.grid(True, alpha=0.3)

# Panel 3 — support-weight CCDF: P(chunk_count ≥ x), log-log. A straight-ish
# line ⇒ heavy-tailed (healthy); a cliff ⇒ one shelf hoards all the support.
ax = axes[1, 0]
weights = sorted((s.chunk_count for s in shelves if s.chunk_count > 0), reverse=True)
if weights:
    ranks = np.arange(1, len(weights) + 1)
    ccdf = ranks / len(weights)
    ax.loglog(weights, ccdf, 'o-', markersize=3, color='darkgreen')
ax.set_title('Support-weight CCDF\n(chunk_count per shelf, rank-ordered)')
ax.set_xlabel('chunk_count (log)')
ax.set_ylabel('P(X ≥ chunk_count) (log)')
ax.grid(True, which='both', alpha=0.3)

# Panel 4 — attachment fan-out: top shelves by real ATTACHED_TO degree.
ax = axes[1, 1]
ranked = sorted(attach_degree.items(), key=lambda kv: -kv[1])[:15]
if ranked:
    labels = [(by_id[sid].label[:26] if sid in by_id else sid)[:26] for sid, _ in ranked]
    vals = [n for _, n in ranked]
    colors = ['crimson' if sid.startswith('facet:') else 'slateblue'
              for sid, _ in ranked]
    y = np.arange(len(labels))
    ax.barh(y, vals, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
src = 'graph edges' if driver is not None else 'projection chunk_count'
ax.set_title(f'Attachment fan-out — top 15 shelves\n(by {src}; '
             f'crimson = synthetic root)')
ax.set_xlabel('attached chunks')
ax.grid(True, axis='x', alpha=0.3)

fig.suptitle(f'Layer A connectivity profile — {FACET or "all facets"}', fontsize=14)
plt.tight_layout()
plt.show()

## 6e. Is Layer A ready for Layer B?

Layer B clusters chunks *within* each shelf into themes. Its hard gate is
`min_chunks_per_shelf` (default 50) — a shelf with fewer attached chunks is
skipped. So "is Layer A good enough" reduces to four questions:

1. **Structurally sound?** — `fs.audit().passed` (single root, no cycles, no
   dangling parents, attachment coverage/integrity). This is the go/no-go gate.
2. **Right-sized?** — how many shelves clear `min_chunks_per_shelf` (Layer B
   will process these), how many are too small (skipped), and is there a
   dumping-ground shelf (esp. the synthetic facet root) that means Layer A
   under-resolved? That's the cell below.
3. **Coherent?** — do a shelf's chunks actually belong to it? Read
   `fs.quality_report()` sections 1 & 5 by eye; off-topic chunks → noisy themes.
4. **Embeddings present?** — Leiden needs chunk vectors; `fs.embed()` must have
   run. The cell checks this too.

Decision: **gate on `audit.passed`, then eyeball** the readiness numbers + the
quality report. There's no single score — coherence is a judgment call.

In [ ]:
# Layer B readiness — true attached-chunk counts per shelf (what Layer B sees),
# via the backend-agnostic attachment map. Run AFTER fs.attach().
from collections import Counter

_min = fs.config.layer_b.min_chunks_per_shelf
_shelves = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_by_id = {s.shelf_id: s for s in _shelves}

# Invert chunk->shelves into shelf->chunk_count (real edges, not projection-time).
_attach = fs.graph_store.list_chunk_shelf_attachments()
_count: Counter = Counter()
for _cid, _sids in _attach.items():
    for _sid in _sids:
        if _sid in _by_id:
            _count[_sid] += 1

# 1. Structural gate.
_audit = fs.audit()
print(f'1. STRUCTURAL GATE  ->  audit.passed = {_audit.passed}')
if not _audit.passed:
    for c in _audit.critical_failures:
        print(f'   CRITICAL: {c.section} {c.name} (metric={c.metric}, thr={c.threshold})')
    print('   Fix criticals before Layer B.\n')

# 2. Size distribution against min_chunks_per_shelf.
_clusterable = [s for s in _shelves if _count[s.shelf_id] >= _min]
_toosmall    = [s for s in _shelves if 0 < _count[s.shelf_id] < _min]
_empty       = [s for s in _shelves if _count[s.shelf_id] == 0]
_total_chunks = sum(_count.values())
print(f'\n2. SIZE (min_chunks_per_shelf = {_min})')
print(f'   clusterable (>= {_min}): {len(_clusterable)} shelves '
      f'-> Layer B will process these')
print(f'   too small (1..{_min-1}):  {len(_toosmall)} shelves -> skipped')
print(f'   empty (0 chunks):       {len(_empty)} shelves')

# Dumping-ground check: any shelf (esp. synthetic root) holding a huge share?
print('\n   biggest shelves by attached chunks:')
for _sid, _n in _count.most_common(8):
    _share = _n / _total_chunks if _total_chunks else 0
    _root = ' (SYNTHETIC ROOT)' if _sid == 'facet:foods' else ''
    print(f'     {_by_id[_sid].label[:38]:38s} {_n:>6d}  ({_share:4.0%}){_root}')
_root_n = _count.get('facet:foods', 0)
if _total_chunks and _root_n / _total_chunks > 0.30:
    print(f'   ⚠ synthetic root holds {_root_n/_total_chunks:.0%} of chunks — Layer A '
          f'under-resolved; many chunks have no specific shelf.')

# 3. Coherence pointer.
print('\n3. COHERENCE  ->  read fs.quality_report(facet=\'foods\') sections 1 & 5')
print('   (do the sample chunks on each shelf actually belong there?)')

# 4. Embeddings present?
_sample = next((c for batch in fs.chunk_store.iter_chunks() for c in batch
                if c.shelf_ids), None)
_has_emb = _sample is not None and _sample.embedding is not None
print(f'\n4. EMBEDDINGS present on attached chunks: {_has_emb}'
      + ('' if _has_emb else '  -> run fs.embed() before Layer B'))

print(f'\nVERDICT: {len(_clusterable)} shelves ready to cluster, '
      f'{_total_chunks} chunks attached. '
      + ('Structurally GO.' if _audit.passed else 'Structural FAIL — fix first.'))

### 6f. Why is 'food product' a dumping ground?

`food product` holds ~10% of attached chunks. It's a generic FoodOn umbrella
class that *should* have been dropped by the umbrella rule (drop iff
`direct_share < 0.10` AND `lifted_share > 0.85`). It survived — this cell shows
which condition it fails, which tells us the fix:

- **High direct_share** → chunks are NEL-linked straight to the generic term
  ('food'/'food product' mentions resolve to `FOODON:00001002`). Fix is upstream
  (linker / link_blocklist), not pruning.
- **Low lifted_share** → it has real descendant support but also too many direct
  hits. Same upstream cause.
- **Below umbrella_min_count** → unlikely at this size.

Note: the 2,536 'attached' chunks include everything that *lifted* to it (chunks
whose specific term was pruned, resolving to the nearest survivor). So part of
this is descendants with no better home — separate from direct mislinking.

In [ ]:
# Diagnose 'food product' against the umbrella thresholds + see what links to it.
_la = fs.config.layer_a
_fp = next((s for s in fs.graph_store.list_shelves()
            if s.facet == 'foods' and s.label == 'food product'), None)
if _fp is None:
    print('no shelf labelled \'food product\' — maybe it was merged/renamed')
else:
    _wd = _fp.chunk_count or (_fp.support_direct + _fp.support_lifted)
    _dshare = _fp.support_direct / _wd if _wd else 0
    _lshare = _fp.support_lifted / _wd if _wd else 0
    print(f'shelf {_fp.shelf_id}  foodon={_fp.foodon_id}  depth={_fp.depth}')
    print(f'  support_direct = {_fp.support_direct}')
    print(f'  support_lifted = {_fp.support_lifted}')
    print(f'  with_descendants (chunk_count) = {_wd}')
    print(f'  direct_share = {_dshare:.3f}   (umbrella drops if < {_la.umbrella_direct_share_max})')
    print(f'  lifted_share = {_lshare:.3f}   (umbrella drops if > {_la.umbrella_lifted_share_min})')
    print(f'  umbrella_min_count = {_la.umbrella_min_count}  (rule only applies if chunk_count >= this)')
    _would_drop = (_dshare < _la.umbrella_direct_share_max
                   and _lshare > _la.umbrella_lifted_share_min
                   and _wd >= _la.umbrella_min_count)
    print(f'\n  -> umbrella rule WOULD drop it: {_would_drop}')
    if not _would_drop:
        if _dshare >= _la.umbrella_direct_share_max:
            print(f'     FAILS on direct_share: {_dshare:.0%} of support is DIRECT mentions.')
            print('     Cause: the linker maps vague mentions to the generic term.')
            print('     Fix: add (surface, FOODON:00001002) pairs to layer_a.link_blocklist,')
            print('          or lower nel confidence for generic surfaces. NOT a pruning fix.')

# What surface forms link to food product? (samples from attached chunks)
from collections import Counter
_fp_id = _fp.foodon_id if _fp else None
if _fp_id:
    _surfaces: Counter = Counter()
    _n = 0
    for _batch in fs.chunk_store.iter_chunks():
        for _c in _batch:
            if _fp_id in (_c.foodon_ids or []):
                _n += 1
                for _link in _c.entity_links:
                    if _link.ontology_id == _fp_id:
                        _surfaces[_link.mention.text.lower()] += 1
            if _n >= 2000: break
        if _n >= 2000: break
    print(f'\n  direct mentions linking to {_fp_id} (top surfaces, sampled):')
    for _surf, _cnt in _surfaces.most_common(15):
        print(f'    {_cnt:>4d}  {_surf!r}')
    if not _surfaces:
        print('    (none in sample — its support is all lifted from descendants,')
        print('     meaning it survives because lifted_share <= threshold or min_count)')

In [ ]:
# WHICH PATH feeds FOODON:00001002 support? The link_blocklist only filters
# `entity_links`; the foods-facet `foodon_ids` denorm path bypasses it. If the
# term arrives via foodon_ids, the blocklist can't drop it.
_TERM = 'FOODON:00001002'
_thr = fs.config.layer_a.min_link_confidence

# Confirm the blocklist actually loaded on the live config:
_bl = {(e.surface.lower(), e.ontology_id) for e in fs.config.layer_a.link_blocklist}
print('blocklist entries for this term:',
      sorted(s for (s, t) in _bl if t == _TERM)[:20])
print(f'min_link_confidence = {_thr}\n')

via_links_only = 0   # term in entity_links, NOT in foodon_ids
via_foodon_ids = 0   # term in foodon_ids (bypasses blocklist)
would_block    = 0   # entity_link whose (surface, term) is blocklisted
both = 0
n = 0
for _batch in fs.chunk_store.iter_chunks():
    for c in _batch:
        in_links = any(
            l.ontology_id == _TERM and l.confidence >= _thr
            for l in c.entity_links
        )
        in_denorm = _TERM in (c.foodon_ids or [])
        if in_links or in_denorm:
            n += 1
            if in_denorm:
                via_foodon_ids += 1
            if in_links and not in_denorm:
                via_links_only += 1
            if in_links and in_denorm:
                both += 1
            for l in c.entity_links:
                if l.ontology_id == _TERM and (l.mention.text.lower(), _TERM) in _bl:
                    would_block += 1
                    break
        if n >= 5000:
            break
    if n >= 5000:
        break

print(f'chunks touching {_TERM} (sampled {n}):')
print(f'  via foodon_ids denorm (BYPASSES blocklist): {via_foodon_ids}')
print(f'  via entity_links only (blocklist applies):  {via_links_only}')
print(f'  via both:                                   {both}')
print(f'  entity_links that the blocklist WOULD drop: {would_block}')
print()
if via_foodon_ids > via_links_only:
    print('=> Support comes mostly via foodon_ids -> the link_blocklist CANNOT')
    print('   fix this. Need a foodon_ids-aware filter in collect_support.')
else:
    print('=> Support comes via entity_links -> blocklist should work after a')
    print('   real re-projection. Verify build_layer_a actually re-ran.')

### 6g. Why are 18% of chunks orphaned at the synthetic root?

Dropping 'food product' sent its chunks looking for a new home. Those whose
specific FoodOn term — and *every surviving ancestor* of it — was pruned fall
through to the synthetic root `facet:foods`. This cell breaks down the orphans:

- **top orphan terms** — what FoodOn ids the root's chunks actually link to.
- **best catch-all ancestors** — pruned ancestors ranked by how many orphan
  chunks each would recover if kept as a shelf. A few high-count ancestors at
  the top => whitelist them (so those chunks get a real mid-level home).
  Whether they'd survive a lower `min_support` can't be read from the built
  graph (pruned terms carry no persisted count), so whitelisting is the
  deterministic lever.
- vs **spread thin** — if recovery scatters across many low-count ancestors,
  it's a genuine long-tail; accept it and move to Layer B.

In [ ]:
# Diagnose synthetic-root orphans: what they link to + best catch-all ancestors.
from collections import Counter

_root_id = 'facet:foods'
_shelves = fs.graph_store.list_shelves()
# A term reaches a real shelf if it (or an ancestor) survived as a shelf or was
# folded into one (see_also).
_resolvable = {s.foodon_id for s in _shelves if s.facet == 'foods' and s.foodon_id}
_resolvable |= {fid for s in _shelves if s.facet == 'foods' for fid in s.see_also}

_attach = fs.graph_store.list_chunk_shelf_attachments()
_root_chunk_ids = {cid for cid, sids in _attach.items() if _root_id in sids}
print(f'{len(_root_chunk_ids)} chunks attached to {_root_id}')

_orphan_terms: Counter = Counter()
_seen = 0
for _batch in fs.chunk_store.iter_chunks():
    for c in _batch:
        if c.chunk_id in _root_chunk_ids:
            for fid in (c.foodon_ids or []):
                if fid in fs.ontology and fid not in _resolvable:
                    _orphan_terms[fid] += 1
            _seen += 1
        if _seen >= len(_root_chunk_ids): break
    if _seen >= len(_root_chunk_ids): break

print('\ntop orphan terms (chunks -> term, label):')
for fid, cnt in _orphan_terms.most_common(20):
    print(f'  {cnt:>4d}  {fid}  {fs.ontology.id_to_label(fid)!r}')

# For each orphan term, every pruned ancestor is a candidate catch-all shelf.
# Rank ancestors by total orphan-chunks they'd recover if whitelisted.
_anc_recover: Counter = Counter()
for fid, cnt in _orphan_terms.items():
    for anc in fs.ontology.id_to_ancestors(fid):
        if anc not in _resolvable:        # ancestor was also pruned
            _anc_recover[anc] += cnt

print(f'\nbest catch-all ancestors (whitelist candidates) — min_support={fs.config.layer_a.min_support}:')
for anc, recoverable in _anc_recover.most_common(15):
    print(f'  +{recoverable:>4d} chunks  {anc}  {fs.ontology.id_to_label(anc)!r}')
print('\nFew high-count ancestors at the top => whitelist them via')
print('layer_a.facet_overrides[\'foods\'].whitelist. Spread thin => long-tail; proceed.')

In [ ]:
# WHY is each real-food orphan orphaned? Trace the prune mechanism per term,
# so we pick the lever that actually helps (min_support vs max_depth vs parent
# pruned) instead of guessing. Recompute the foods support table fresh.
from foodscholar.layer_a.propagate import collect_support

_fc = fs.config.layer_a.resolve_facet('foods')
def _chunks():
    for b in fs.chunk_store.iter_chunks():
        yield from b
_support = collect_support(_chunks(), fs.ontology,
                           min_link_confidence=_fc.min_link_confidence,
                           facet='foods', link_blocklist=_fc.link_blocklist)

_shelves = fs.graph_store.list_shelves()
_surv = {s.foodon_id for s in _shelves if s.facet=='foods' and s.foodon_id}
_surv |= {fid for s in _shelves if s.facet=='foods' for fid in s.see_also}
_min = fs.config.layer_a.min_support
_cap = fs.config.layer_a.max_depth

# Probe the real-food orphans seen in 6g.
_probe = ['FOODON:00004387','FOODON:03411043','FOODON:00004777','FOODON:00004560',
          'FOODON:03311555','FOODON:03310371']
print(f'min_support={_min}  max_depth={_cap}\n')
for fid in _probe:
    lbl = fs.ontology.id_to_label(fid)
    wd = _support.with_descendants.get(fid, 0)
    direct = _support.direct.get(fid, 0)
    # Which ancestors had enough support to survive the threshold?
    anc_surv = [(a, _support.with_descendants.get(a,0)) for a in fs.ontology.id_to_ancestors(fid)]
    anc_over = [(a,n) for a,n in anc_surv if n >= _min]
    reason = []
    if wd < _min: reason.append(f'self below min_support ({wd}<{_min})')
    else: reason.append(f'self HAS support ({wd}>={_min}) -> not a threshold victim')
    if not any(a in _surv for a in fs.ontology.id_to_ancestors(fid)):
        reason.append('NO ancestor survived as a shelf')
    print(f'{fid} {lbl!r}: direct={direct} with_desc={wd}')
    print(f'   reason: {"; ".join(reason)}')
    if anc_over[:3]:
        print(f'   ancestors over min_support (would-be parents): '
              + ', '.join(f'{fs.ontology.id_to_label(a)!r}({n})' for a,n in anc_over[:3]))
    print()
print('If self HAS support but is still orphaned -> max_depth lifted it past a')
print('surviving ancestor (lower-priority). If self is below min_support AND an')
print('ancestor is over -> lowering min_support or whitelisting that ancestor helps.')

### 6h. Which mid-level categories to whitelist?

Rare foods orphan because their only ancestors are umbrellas. Whitelisting a few
*specific* mid-level food categories gives them a home — but we must avoid the
too-generic ones ('food product', 'food material', BFO 'material entity') that
would just rebuild the dumping ground.

This cell ranks candidate parents by how many **distinct real-food orphan
terms** each would adopt, prefers the **deepest** (most specific) candidate, and
runs a greedy set-cover to find the **minimal whitelist** that re-homes the most
orphans. Copy the printed `whitelist=[...]` into the config and re-project.

In [ ]:
# Recommend mid-level whitelist by SPECIFICITY: assign each orphan to its
# DEEPEST non-generic ancestor, then count orphans per category. This avoids
# broad umbrellas (greedy-by-coverage picked 'plant food product' etc.).
from foodscholar.layer_a.propagate import collect_support
from collections import Counter

# A category is too generic to be a useful parent if it's a FoodOn
# classification AXIS (metadata, not a food kind) or a top-tier umbrella.
_GENERIC_SUBSTRINGS = (' by ', 'material', 'edible food', 'food product',
                       'claim', 'consumer group', 'characteristic',
                       'organism', 'multi-component', 'food (')
def _too_generic(fid):
    if not fid.startswith('FOODON:'):   # drops BFO:* etc.
        return True
    lbl = (fs.ontology.id_to_label(fid) or '').lower()
    # bare 'food product' / 'plant food product' / 'animal food product' tier
    if lbl.endswith('food product') and lbl.count(' ') <= 2:
        return True
    return any(sub in lbl for sub in _GENERIC_SUBSTRINGS)

_fc = fs.config.layer_a.resolve_facet('foods')
def _chunks():
    for b in fs.chunk_store.iter_chunks():
        yield from b
_support = collect_support(_chunks(), fs.ontology,
                           min_link_confidence=_fc.min_link_confidence,
                           facet='foods', link_blocklist=_fc.link_blocklist)
_min = fs.config.layer_a.min_support
_shelves = fs.graph_store.list_shelves()
_surv = {s.foodon_id for s in _shelves if s.facet=='foods' and s.foodon_id}
_surv |= {fid for s in _shelves if s.facet=='foods' for fid in s.see_also}

# Real-food orphans: own support, not generic, no surviving ancestor.
_orphans = [fid for fid in _support.direct.keys()
            if _support.with_descendants.get(fid,0) >= _min
            and not _too_generic(fid)
            and fid not in _surv
            and not any(a in _surv for a in fs.ontology.id_to_ancestors(fid))]
print(f'{len(_orphans)} real-food orphan terms\n')

# id_to_ancestors is closed & ordered nearest-first? Don't assume — rank
# candidate parents of an orphan by ontology depth (deeper = more specific).
def _depth(fid):
    return len(fs.ontology.id_to_ancestors(fid))  # proxy: more ancestors = deeper

_assigned = Counter()       # category -> # orphans whose deepest non-generic parent it is
_adopts = {}                # category -> list of example orphans
_uncovered = []
for fid in _orphans:
    cands = [a for a in fs.ontology.id_to_ancestors(fid) if not _too_generic(a)]
    if not cands:
        _uncovered.append(fid); continue
    parent = max(cands, key=_depth)      # deepest = most specific
    _assigned[parent] += 1
    _adopts.setdefault(parent, []).append(fid)

print(f'specific mid-level parents (deepest non-generic), '
      f'{len(_orphans)-len(_uncovered)}/{len(_orphans)} orphans covered:')
for cat, n in _assigned.most_common(20):
    egs = ', '.join(fs.ontology.id_to_label(o) or o for o in _adopts[cat][:3])
    print(f'  +{n:>3d}  {cat}  {fs.ontology.id_to_label(cat)!r}   e.g. {egs}')
print(f'\nlong-tail with no specific parent: {len(_uncovered)} terms')

# Suggest whitelisting categories that adopt >= 3 orphans (worth a shelf).
_picks = [(c,n) for c,n in _assigned.most_common() if n >= 3]
print('\n# whitelist categories adopting >= 3 orphans:')
print('"foods": {"whitelist": [')
for fid,_ in _picks:
    print(f'    "{fid}",  # {fs.ontology.id_to_label(fid)}')
print(']}')

## 6d. Semantic consolidation (LLM-as-judge)

Rule-based collapse only catches *structural* duplicates (single-child chains). The ~5% of duplicate shelves that share **meaning but not lexical stem** — `fermented dairy product` vs `yogurt`, `ground cattle meat` vs `beef (ground)` — survive. `fs.semantic_consolidate()` embeds each shelf (label + FoodOn synonyms), finds near-duplicate **pairs** by cosine similarity, **clusters** those pairs into connected components, then asks an LLM judge — **one call per cluster** — to decide which members merge and which stay distinct, grounded in real sample chunks pulled from the store.

Clustering means a 5-shelf cereal group becomes *one* clean decision instead of ~7 fragmented pairwise verdicts (≈5× fewer LLM calls, and transitive closure happens inside the model). Two precision controls:

- **`use_few_shot`** (default on): calibration examples in the prompt that anchor the judge on meaningful distinctions (whole vs skim milk, oil vs fat) — the cheapest fix for an over-merging model.
- **`permanent_block_list`**: pairs of FoodOn ids that must *never* merge (oil/fat, olive-oil/vegetable-oil). A vetoed pair blocks its whole cluster from merging. Grow it whenever a bad merge slips through.

`auto_merge_confidence` defaults to **0.80** (instruct models report bimodal confidence — 0.80 is their "yes"). It runs **after** `fs.attach()` and is **off by default**. Embedder is `fs.embedder` (BGE-large); judge is `fs.llm`.

Workflow: **(1)** preview candidates (no LLM) → **(2)** dry-run the judge → **(3)** persist.

In [8]:
# Swap the mock LLM for a real judge. The notebook's from_config had no
# `llm` block, so fs.llm defaulted to the built-in mock. Groq needs
# GROQ_API_KEY in your environment.
import os
from foodscholar.config import LLMConfig, ProviderConfig
from foodscholar.llm import build_llm
os.environ['GROQ_API_KEY'] = '***REMOVED-GROQ-KEY***'
assert os.environ.get('GROQ_API_KEY'), 'set GROQ_API_KEY first'
fs.llm = build_llm(LLMConfig(
    primary=ProviderConfig(provider='groq', model='openai/gpt-oss-20b'),
))
print('judge:', fs.llm.model_id)  # must NOT contain "mock"
print(fs.llm.generate('Reply with the single word: ready'))

judge: openai/gpt-oss-20b


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


ready


In [ ]:
# Threshold sweep (no LLM). Embed once, then for each threshold show how many
# candidate pairs survive AND how they cluster — the cluster sizes are what
# actually matter: too loose a threshold chains everything into one hairball.
from foodscholar.layer_a.semantic_consolidation.candidates import find_candidates
from foodscholar.layer_a.semantic_consolidation.cluster import cluster_candidates
from foodscholar.layer_a.semantic_consolidation.embed import embed_shelves

_sc = fs.config.layer_a.semantic_consolidation
_shelves = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_by_id = {s.shelf_id: s for s in _shelves}
_embs = embed_shelves(_shelves, fs.ontology, fs.embedder, _sc)  # ~230 vectors, once
print(f'{len(_shelves)} foods shelves, {len(_embs)} embedded  '
      f'(cluster cap = {_sc.max_cluster_size})\n')

_orig = _sc.cosine_threshold
print(f'{"thresh":>6}  {"pairs":>6}  {"clusters":>8}  {"biggest":>7}  {"shelves judged":>14}')
for thr in (0.88, 0.90, 0.91, 0.92, 0.93, 0.94, 0.95):
    _sc.cosine_threshold = thr
    cand, _ = find_candidates(_embs, _by_id, _sc)
    clusters = cluster_candidates(cand, _sc.max_cluster_size)
    biggest = max((len(c) for c in clusters), default=0)
    judged = sum(len(c) for c in clusters)
    print(f'{thr:>6.2f}  {len(cand):>6d}  {len(clusters):>8d}  {biggest:>7d}  {judged:>14d}')
_sc.cosine_threshold = _orig  # restore

# Pick the threshold where 'biggest' is small (<= cap, ideally well under) and
# 'clusters' looks like real duplicate groups, not a single blob. Then set it:
# fs.config.layer_a.semantic_consolidation.cosine_threshold = 0.93

In [ ]:
# (1) Zero-cost preview — no LLM. Shows the scaffolding filter, candidates,
# how they cluster, and what the rule filters dropped. Judge off:
fs.config.layer_a.semantic_consolidation.judge_enabled = False

# What the scaffolding filter removes (FoodOn umbrella terms: no synonyms +
# classifier suffix). These never reach the judge. If you spot a REAL food
# here, prune the offending word from cfg.classifier_suffixes (the broad ones
# are 'food'/'foods'/'ingredient').
from foodscholar.layer_a.semantic_consolidation.embed import is_scaffolding
_sc = fs.config.layer_a.semantic_consolidation
_foods = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_scaff = [s for s in _foods if s.foodon_id and is_scaffolding(s, fs.ontology, _sc)]
print(f'scaffolding excluded: {len(_scaff)} of {len(_foods)} foods shelves')
for s in _scaff[:20]:
    print(f'  - {s.label!r}')

art = fs.semantic_consolidate(facet='foods', dry_run=True)
print()
print(art)
print(f'\n{art.candidate_count} candidate pair(s) -> {art.cluster_count} cluster(s); '
      f'{len(art.filtered_pairs)} dropped by subtype/compound filters')
for p in art.filtered_pairs[:10]:
    print(f'  filtered {p.shelf_a} ~ {p.shelf_b} '
          f'(cos={p.cosine_similarity:.3f}): {p.filtered_reason}')

In [ ]:
# DIAGNOSTIC: send ONE real cluster to the judge and show raw + parsed output.
# IMPORTANT: force-reload the sc modules so kernel edits take effect WITHOUT a
# full restart. (A running kernel caches imports — editing the .py on disk is
# not enough; this is almost certainly why earlier results didn't change.)
import importlib, foodscholar.layer_a.semantic_consolidation as _scpkg
import foodscholar.layer_a.semantic_consolidation.prompts as _p
import foodscholar.layer_a.semantic_consolidation.embed as _e
import foodscholar.layer_a.semantic_consolidation.candidates as _c
import foodscholar.layer_a.semantic_consolidation.cluster as _cl
import foodscholar.layer_a.semantic_consolidation.judge as _j
import foodscholar.layer_a.semantic_consolidation.apply as _a
import foodscholar.llm.providers as _prov
for _m in (_p, _e, _c, _cl, _j, _a, _prov, _scpkg):
    importlib.reload(_m)
print('reloaded sc modules; prompt version =', _p.PROMPT_VERSION)
# rebuild the groq client so the reloaded GroqClient.generate_json is used:
from foodscholar.config import LLMConfig, ProviderConfig
from foodscholar.llm import build_llm
import os
if os.environ.get('GROQ_API_KEY'):
    fs.llm = build_llm(LLMConfig(primary=ProviderConfig(
        provider='groq', model='llama-3.3-70b-versatile')))
print('judge:', fs.llm.model_id)

_sc = fs.config.layer_a.semantic_consolidation
_shelves = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_by_id = {s.shelf_id: s for s in _shelves}
_embs = _e.embed_shelves(_shelves, fs.ontology, fs.embedder, _sc)
_cand, _ = _c.find_candidates(_embs, _by_id, _sc)
_clusters = _cl.cluster_candidates(_cand, _sc.max_cluster_size)
print(f'{len(_clusters)} clusters; sizes: {sorted((len(c) for c in _clusters), reverse=True)}')

_cluster = max(_clusters, key=len)
print(f'\njudging cluster of {len(_cluster)}:')
for sid in _cluster:
    print(f'  {sid}  {_by_id[sid].label!r}')

_prompt = _j._build_prompt(_cluster, _by_id, fs.ontology, fs.chunk_store, _sc, {})
_raw = fs.llm.generate_json(_prompt, _p.JUDGE_CLUSTER_SCHEMA,
                            max_tokens=_j._token_budget(len(_cluster)))
print('\n===== RAW MODEL JSON =====')
print(_raw)
_decision = _j._parse_cluster(_cluster, _raw, fs.llm.model_id)
print('\n===== PARSED =====')
print('merge_groups:', [(g.members, g.confidence) for g in _decision.merge_groups])
print('keep_alone  :', len(_decision.keep_alone), 'shelves')
print('\n(if RAW is {} -> groq returned empty; if it has merge_groups but parsed'
      ' is empty -> parse bug; if keep_alone=all -> model declined to merge)')

In [ ]:
# (2) Dry-run WITH the judge (groq/llama via fs.llm). No writes — read the
# proposed merge groups, the uncertain ones, and any block-list vetoes.
fs.config.layer_a.semantic_consolidation.judge_enabled = True

# Permanent block-list: pairs of FoodOn ids that must NEVER merge, by id
# (order-independent). Grow this whenever the judge proposes a bad merge you
# see below. These two were caught on this corpus:
#   cow whole milk vs cow milk      — fat-level subtype, not a duplicate
#   vitamin prep vs vitamin+mineral — category vs broader category
fs.config.layer_a.semantic_consolidation.permanent_block_list = [
    ('FOODON:00005478', 'FOODON:02020891'),   # cow whole milk / cow milk
    ('FOODON:03310620', 'FOODON:03315201'),   # vitamin prep / vitamin+mineral prep
]

art = fs.semantic_consolidate(facet='foods', dry_run=True)
_thr = fs.config.layer_a.semantic_consolidation.auto_merge_confidence

print(f'applied groups (conf >= {_thr}) — would remove {art.shelves_removed} shelves:')
for g in art.applied_groups:
    print(f'  MERGE {g.members} -> {g.canonical_name!r} '
          f'(conf={g.confidence:.2f}): {g.rationale}')

print('\nuncertain groups (below threshold — NOT applied, review by hand):')
for g in art.uncertain_groups:
    print(f'  {g.members} -> {g.canonical_name!r} (conf={g.confidence:.2f}): {g.rationale}')

if art.blocked_groups:
    print('\nblocked by permanent_block_list:')
    for g in art.blocked_groups:
        print(f'  {g.members}  vetoed pairs: {g.blocked_pairs}')

# To veto another bad merge, append its FoodOn-id pair above and re-run.

In [ ]:
# (3) Persist. Applies confirmed groups (N-way), re-upserts the shelf set,
# and re-runs fs.attach() so folded shelves' chunks re-home onto the surviving
# canonical via its see_also list. Re-run audit/quality after.
#
  #set this once you're happy with the dry-run above:
fs.config.layer_a.semantic_consolidation.enabled = True
before = len([s for s in fs.graph_store.list_shelves() if s.facet == 'foods'])
art = fs.semantic_consolidate(facet='foods', dry_run=False)
after = len([s for s in fs.graph_store.list_shelves() if s.facet == 'foods'])
print(f'foods shelves: {before} -> {after}  '
      f'({len(art.applied_groups)} groups applied, -{art.shelves_removed})')
print(fs.audit())

## 7. Build Layer B (themes)

Per-shelf theme discovery via two parallel community detections, then a greedy merge.

- **Pass 1 — Similarity** clusters chunks that *discuss the same topic in similar prose* (mutual-kNN over the embeddings we just built; edges = cosine).
- **Pass 2 — Relatedness** (SiReRAG-inspired) clusters chunks that *co-mention the same FoodOn entities* (edges weighted IDF-style over shared `ontology_id`s; `tau_strict` floors low-confidence links; `max_doc_frequency` drops ubiquitous entities).
- **Merge** pairs `(S_i, R_j)` candidates; combined Jaccard over chunks+entities `≥ dedupe_threshold` (default 0.70) collapses them into a `discovery_pass="merged"` theme. Unmerged candidates pass through as single-pass themes.

The merge step is the **stronger-evidence upgrade**, not deduplication — a merged theme is grounded in *both* topical and entity coherence. If `relatedness` contributes 0 themes, the entity graph is mis-tuned (try `min_shared_ids=1` or `max_doc_frequency=0.6`); if every theme is merged, Pass 2 isn't earning compute.

Required state: chunks have embeddings (you've just finished this) and `fs.attach()` has populated Neo4j HAS_CHUNK edges. Re-run `fs.attach()` first if Neo4j drift is suspected.


In [ ]:
# Pre-build preview — see what Pass 2 (per-shelf relatedness) looks like on
# the largest shelf, and what Pass 1 (global similarity) sees across the whole
# attached corpus. Doesn't run the full pipeline; doesn't write.

from foodscholar.layer_b.builder import (
    build_shelf_relatedness_candidates,
    build_global_similarity_candidates,
)

cfg = fs.config.layer_b

# Pass 2 preview — pick the busiest shelf, build relatedness candidates.
attachments = fs.graph_store.list_chunk_shelf_attachments()
shelf_to_chunks = {}
for chunk_id, shelf_ids in attachments.items():
    for sid in shelf_ids:
        shelf_to_chunks.setdefault(sid, []).append(chunk_id)

eligible = [
    (sid, len(cids)) for sid, cids in shelf_to_chunks.items()
    if sid != "facet:foods" and len(cids) >= cfg.min_chunks_per_shelf
]
eligible.sort(key=lambda x: -x[1])
preview_shelf, n_chunks = eligible[0]
print(f"--- Pass 2 (relatedness) preview ---")
print(f"shelf: {preview_shelf!r}  attached_chunks={n_chunks}")

chunks = fs.chunk_store.get_many(shelf_to_chunks[preview_shelf])
embedded = [c for c in chunks if c.embedding is not None]
print(f"  with embeddings: {len(embedded)}/{len(chunks)} ({len(embedded)/len(chunks):.1%})")
rel_cands = build_shelf_relatedness_candidates(chunks, cfg)
print(f"  relatedness candidates: {len(rel_cands):>3}  sizes={sorted([len(c.chunk_ids) for c in rel_cands], reverse=True)[:10]}")

# Pass 1 preview — global similarity across ALL attached chunks. Issues one ES
# kNN per chunk; on ~6k chunks expect ~30-60s. Skip if you've already run §7c.
print(f"\n--- Pass 1 (global_similarity) preview ---")
synth_root = "facet:foods"
facet_shelves = {s.shelf_id for s in fs.graph_store.list_shelves() if s.facet == "foods"}
attached_chunk_ids = sorted({
    cid for cid, sids in attachments.items()
    if any(sid in facet_shelves and sid != synth_root for sid in sids)
})
print(f"global pass would see {len(attached_chunk_ids)} attached chunks")
if len(attached_chunk_ids) > cfg.global_similarity_max_chunks:
    print(f"  (exceeds cfg.global_similarity_max_chunks={cfg.global_similarity_max_chunks}; "
          f"orchestrator would skip global pass)")
else:
    glob_cands = build_global_similarity_candidates(attached_chunk_ids, fs.chunk_store, cfg)
    print(f"  global_similarity candidates: {len(glob_cands):>3}  "
          f"sizes={sorted([len(c.chunk_ids) for c in glob_cands], reverse=True)[:10]}")


In [ ]:
# Full Layer B rollout. Run with dry_run=True first to see the artifact shape
# without writing; flip dry_run=False to persist.
#
# Cost: ~10-15 minutes wall-clock for ~100-shelf rollouts; ~$0.60 LLM cost
# at default labeling.strategy="llm".

DRY_RUN = True  # flip to False once you've inspected the dry-run output

artifact = fs.build_layer_b(facet="foods", dry_run=DRY_RUN)
print(artifact.model_dump_json(indent=2))


### 7b. Tune Layer B knobs in the notebook

Three knobs shape the global Pass 1 (cross-shelf similarity) and the Leiden community split:

- `similarity.knn_k` — how many neighbors each chunk sees. Higher = denser graph, larger communities. Lower = sparser graph, more (smaller) communities.
- `similarity.edge_threshold` — cosine cutoff below which neighbor edges are dropped. Higher = only tight pairs, more (smaller) communities. Lower = looser graph, fewer (larger) communities.
- `leiden.resolution` — Leiden's resolution parameter. Higher = more, smaller communities. Lower = fewer, larger ones.

Edit the values below, then rerun the Build cell (§7) to see the effect. The defaults baked into `LayerBConfig` are `knn_k=15, edge_threshold=0.5, leiden.resolution=1.0` — the first build at those defaults produced a single 213-chunk megacluster, so the §7c sweep below picks tighter values automatically.


In [ ]:
# In-notebook knob overrides. Tune these, then rerun the §7 build cell.
# Comment a line out to keep the LayerBConfig default.

fs.config.layer_b.similarity.knn_k = 8           # was 15 — sparser graph
fs.config.layer_b.similarity.edge_threshold = 0.65  # was 0.50 — tighter pairs
fs.config.layer_b.leiden.resolution = 1.5        # was 1.0 — more communities

print(f"knn_k             = {fs.config.layer_b.similarity.knn_k}")
print(f"edge_threshold    = {fs.config.layer_b.similarity.edge_threshold}")
print(f"leiden.resolution = {fs.config.layer_b.leiden.resolution}")


### 7c. Knob sweep — what values minimize megaclusters?

A real build is several minutes; a 7×3 sweep at that cost is hours. Instead, this cell:

1. Pre-fetches every attached-chunk embedding ONCE (the only ES round-trip).
2. For each knob value, re-runs `build_global_similarity_graph` + Leiden purely in-memory against the cached embeddings — ~10-30s per point.
3. Plots theme count, median theme size, largest theme size, and cross-shelf fraction vs. each knob.

The "good" region is where you avoid mega-themes (largest_size < 5% of corpus) without fragmenting into too many tiny themes. Marked dot = currently configured value; vertical dashed line = the proposed sweet spot.


In [ ]:
# Layer B Pass-1 sweep. ~2-5 minutes wall-clock on a 6k-chunk corpus.
# Embeds the chunks once (cached on the chunk store) then runs the sim+Leiden
# pipeline in-process per knob value. No ES per point, no Neo4j writes.

import time
from copy import deepcopy

import igraph as ig
import matplotlib.pyplot as plt
import numpy as np

from foodscholar.layer_b.community import run_leiden
from foodscholar.layer_b.semantic_graph import build_global_similarity_graph

# --- 1. Snapshot attached chunks once ---
FACET = "foods"
attachments = fs.graph_store.list_chunk_shelf_attachments()
facet_shelves = {s.shelf_id for s in fs.graph_store.list_shelves() if s.facet == FACET}
synth_root = f"facet:{FACET}"
attached_chunk_ids = sorted({
    cid for cid, sids in attachments.items()
    if any(sid in facet_shelves and sid != synth_root for sid in sids)
})
print(f"sweeping {len(attached_chunk_ids)} attached chunks")
print("(this prefetches all chunk embeddings in one ES round-trip per knob "
      "value via the existing chunk_store.get_many path — the global graph "
      "builder will still issue per-chunk kNN calls, but ES kNN is fast)")


# --- 2. Sweep ---
def _summarize(g, communities, n_attached):
    sizes = sorted((len(c) for c in communities), reverse=True)
    if not sizes:
        return {"n_themes": 0, "median_size": 0, "max_size": 0, "edges": g.ecount()}
    return {
        "n_themes": len(sizes),
        "median_size": int(np.median(sizes)),
        "max_size": sizes[0],
        "max_size_pct": sizes[0] / n_attached,
        "edges": g.ecount(),
    }


def _run(knn_k, edge_thr, resolution):
    sim_cfg = deepcopy(fs.config.layer_b.similarity)
    sim_cfg.knn_k = knn_k
    sim_cfg.edge_threshold = edge_thr
    leiden_cfg = deepcopy(fs.config.layer_b.leiden)
    leiden_cfg.resolution = resolution
    t0 = time.time()
    g = build_global_similarity_graph(attached_chunk_ids, fs.chunk_store, sim_cfg)
    communities = run_leiden(g, leiden_cfg)
    dt = time.time() - t0
    summary = _summarize(g, communities, len(attached_chunk_ids))
    summary["seconds"] = round(dt, 1)
    return summary


# 1-D sweeps — vary one knob at a time around the configured value
KNN_AXIS = [4, 6, 8, 10, 12, 15, 20]
THR_AXIS = [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
RES_AXIS = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5]

# Hold the OTHER two at the current notebook overrides
cur_knn = fs.config.layer_b.similarity.knn_k
cur_thr = fs.config.layer_b.similarity.edge_threshold
cur_res = fs.config.layer_b.leiden.resolution

print(f"\nSweep 1: knn_k @ thr={cur_thr}, res={cur_res}")
knn_results = [_run(k, cur_thr, cur_res) for k in KNN_AXIS]
for k, r in zip(KNN_AXIS, knn_results):
    print(f"  knn={k:2d} → themes={r['n_themes']:3d}  median={r['median_size']:4d}  "
          f"max={r['max_size']:4d} ({r['max_size_pct']:.0%})  edges={r['edges']:5d}  "
          f"[{r['seconds']:.1f}s]")

print(f"\nSweep 2: edge_threshold @ knn={cur_knn}, res={cur_res}")
thr_results = [_run(cur_knn, t, cur_res) for t in THR_AXIS]
for t, r in zip(THR_AXIS, thr_results):
    print(f"  thr={t:.2f} → themes={r['n_themes']:3d}  median={r['median_size']:4d}  "
          f"max={r['max_size']:4d} ({r['max_size_pct']:.0%})  edges={r['edges']:5d}  "
          f"[{r['seconds']:.1f}s]")

print(f"\nSweep 3: leiden.resolution @ knn={cur_knn}, thr={cur_thr}")
# Resolution sweep can reuse a single graph — only Leiden changes.
sim_cfg = deepcopy(fs.config.layer_b.similarity)
sim_cfg.knn_k = cur_knn
sim_cfg.edge_threshold = cur_thr
g_shared = build_global_similarity_graph(attached_chunk_ids, fs.chunk_store, sim_cfg)
res_results = []
for r_val in RES_AXIS:
    leiden_cfg = deepcopy(fs.config.layer_b.leiden)
    leiden_cfg.resolution = r_val
    t0 = time.time()
    comms = run_leiden(g_shared, leiden_cfg)
    res_results.append({**_summarize(g_shared, comms, len(attached_chunk_ids)),
                        "seconds": round(time.time() - t0, 1)})
for r_val, r in zip(RES_AXIS, res_results):
    print(f"  res={r_val:.1f} → themes={r['n_themes']:3d}  median={r['median_size']:4d}  "
          f"max={r['max_size']:4d} ({r['max_size_pct']:.0%})  [{r['seconds']:.1f}s]")


# --- 3. Plot the three sweeps in a single 3x2 grid ---
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
fig.suptitle("Layer B Pass-1 knob sweeps", fontsize=14)

def _plot(ax, axis, results, key, ylabel, title, current_val):
    ax.plot(axis, [r[key] for r in results], "o-", lw=2)
    ax.axvline(current_val, color="grey", ls="--", alpha=0.6,
               label=f"current = {current_val}")
    ax.set_xlabel(title.split(" — ")[0])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)


_plot(axes[0, 0], KNN_AXIS, knn_results, "n_themes",
      "themes", "knn_k — theme count", cur_knn)
_plot(axes[0, 1], KNN_AXIS, knn_results, "max_size",
      "largest theme size", "knn_k — max theme size", cur_knn)
_plot(axes[1, 0], THR_AXIS, thr_results, "n_themes",
      "themes", "edge_threshold — theme count", cur_thr)
_plot(axes[1, 1], THR_AXIS, thr_results, "max_size",
      "largest theme size", "edge_threshold — max theme size", cur_thr)
_plot(axes[2, 0], RES_AXIS, res_results, "n_themes",
      "themes", "leiden.resolution — theme count", cur_res)
_plot(axes[2, 1], RES_AXIS, res_results, "max_size",
      "largest theme size", "leiden.resolution — max theme size", cur_res)

# Mark "megacluster" line at 5% of corpus on the max-size plots
megaline = 0.05 * len(attached_chunk_ids)
for ax in (axes[0, 1], axes[1, 1], axes[2, 1]):
    ax.axhline(megaline, color="red", ls=":", alpha=0.6,
               label=f"5% of corpus = {int(megaline)}")
    ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

# --- 4. Recommendation ---
def _pick_sweet_spot(axis, results, n_attached):
    """Pick the largest theme-count point whose max_size is below the
    5%-of-corpus megacluster line. Falls back to the min-max-size point."""
    megaline = 0.05 * n_attached
    ok = [(a, r) for a, r in zip(axis, results) if r["max_size"] <= megaline]
    if ok:
        return max(ok, key=lambda x: x[1]["n_themes"])[0]
    return min(zip(axis, results), key=lambda x: x[1]["max_size"])[0]


reco_knn = _pick_sweet_spot(KNN_AXIS, knn_results, len(attached_chunk_ids))
reco_thr = _pick_sweet_spot(THR_AXIS, thr_results, len(attached_chunk_ids))
reco_res = _pick_sweet_spot(RES_AXIS, res_results, len(attached_chunk_ids))
print(f"\nproposed sweet spot:")
print(f"  fs.config.layer_b.similarity.knn_k           = {reco_knn}")
print(f"  fs.config.layer_b.similarity.edge_threshold  = {reco_thr}")
print(f"  fs.config.layer_b.leiden.resolution          = {reco_res}")
print("\napply with the §7b cell above, then rerun the §7 build.")


### 7d. Retrospective — theme shape after the build

After running §7 with the new knobs, this cell answers: did the tune actually reshape the themes? Three views:

- **theme size histogram** — peaked at small sizes is good; a long right tail is the megacluster signature
- **shelves-per-theme histogram** — cross-shelf themes (≥2) are the whole point of `global_similarity`; many themes pinned to a single shelf means Pass 1 didn't bridge anything
- **theme-count by pass** — sanity check on the `global_similarity` / `relatedness` / `merged` split


In [ ]:
# Retrospective post-build charts. Run AFTER §7 with the new knobs.
from collections import Counter

import matplotlib.pyplot as plt

themes = fs.graph_store.list_themes()
themes = [t for t in themes if t.facet == "foods"]

sizes = sorted([t.chunk_count for t in themes], reverse=True)
shelves_per_theme = [len(t.shelf_ids) for t in themes]
by_pass = Counter(t.discovery_pass for t in themes)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (1) theme size histogram (log y, log-ish bins so the tail is visible)
axes[0].hist(sizes, bins=30)
axes[0].set_yscale("log")
axes[0].axvline(0.05 * sum(sizes) / max(1, len(sizes)) * 20, color="red", ls=":",
                alpha=0.6, label="5% of corpus")
axes[0].set_xlabel("chunks per theme")
axes[0].set_ylabel("themes (log)")
axes[0].set_title(f"theme size — n={len(themes)}, max={max(sizes) if sizes else 0}")
axes[0].grid(alpha=0.3)

# (2) shelves per theme
shelves_counter = Counter(shelves_per_theme)
xs = sorted(shelves_counter)
axes[1].bar(xs, [shelves_counter[x] for x in xs])
axes[1].set_xlabel("shelves attached to theme")
axes[1].set_ylabel("themes")
cross = sum(c for n, c in shelves_counter.items() if n >= 2)
axes[1].set_title(f"shelves per theme — cross-shelf = {cross}/{len(themes)}")
axes[1].grid(alpha=0.3, axis="y")

# (3) themes by pass
passes = list(by_pass)
axes[2].bar(passes, [by_pass[p] for p in passes])
axes[2].set_xlabel("discovery_pass")
axes[2].set_ylabel("themes")
axes[2].set_title("themes by pass")
axes[2].grid(alpha=0.3, axis="y")
for i, p in enumerate(passes):
    axes[2].text(i, by_pass[p], str(by_pass[p]), ha="center", va="bottom")

plt.tight_layout()
plt.show()

# Largest themes (top 10) — useful for eyeballing megaclusters
print("\nTop 10 largest themes:")
top = sorted(themes, key=lambda t: -t.chunk_count)[:10]
for t in top:
    print(f"  {t.chunk_count:4d} chunks, {len(t.shelf_ids):2d} shelves, "
          f"[{t.discovery_pass:18s}] {t.label!r}")


In [ ]:
# Cross-store audit + per-pass tuning canaries. Run AFTER a live build.
# `report.passed` enforces the CRITICAL invariants (parity = 1.0, no dangling
# theme_ids, no empty themes). The per-pass + merged_rate fields are WARN-level
# tuning feedback — they don't flip `passed`.

from foodscholar.layer_b.audit import audit_layer_b

report = audit_layer_b(fs.chunk_store, fs.graph_store)
print(report.model_dump_json(indent=2))

# Canary checks (won't fail the audit but worth flagging in the notebook):
if report.by_pass.get("relatedness", 0) == 0:
    print("\nWARN: 0 relatedness themes — Pass 2 found no entity-coherent clusters.")
    print("      Try cfg.layer_b.relatedness.min_shared_ids=1 or max_doc_frequency=0.6.")
if report.merged_rate >= cfg.layer_b.audit.merged_rate_max:
    print(f"\nWARN: merged_rate={report.merged_rate:.2f} is high — Pass 2 may not be earning compute.")
elif report.merged_rate <= cfg.layer_b.audit.merged_rate_min and report.merged_rate > 0:
    print(f"\nWARN: merged_rate={report.merged_rate:.2f} is low — the two passes barely overlap.")


In [ ]:
# §17 sanity gate — sample 20 random themes, print label + 3 chunk excerpts.
# Target ≥ 75% coherent by eye (label names something specific; chunks share
# that focus). Use this to decide whether the run is shippable.

import random

themes = fs.graph_store.list_themes()
print(f"total themes: {len(themes)}")

rng = random.Random(42)  # deterministic sample so audit is reproducible
sample = rng.sample(themes, min(20, len(themes)))

for i, t in enumerate(sample, 1):
    print(f"\n--- {i:>2}. [{t.discovery_pass:>11}] {t.label}")
    print(f"     id: {t.theme_id}")
    print(f"     keywords: {t.keyword_terms[:5]}")
    print(f"     foodon: {t.foodon_id_signature[:5]}")
    chunk_ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
    print(f"     chunks: {len(chunk_ids)}")
    sample_chunks = fs.chunk_store.get_many(chunk_ids[:3])
    for c in sample_chunks:
        excerpt = c.text[:160].replace("\n", " ")
        print(f"       - [{c.chunk_id}] {excerpt}...")


## 8. Inspect


In [ ]:
# A handful of chunks with their attached mentions / foodon ids.
for c in fs.chunk_store.scan()[:3]:
    print(f"\n[{c.chunk_id}] {c.source_type} — {c.text[:80]}...")
    print(f"  mentions:   {[m.text for m in c.mentions[:6]]}")
    print(f"  foodon_ids: {c.foodon_ids[:6]}")
    other = sorted({ln.ontology_id for ln in c.entity_links if not ln.ontology_id.startswith('FOODON:')})
    print(f"  other_links: {other[:6]}")

# BM25 round-trip against Elastic.
hits = fs.chunk_store.search("olive oil", k=3, use_vector=False)
print("\n'olive oil' top 3:")
for h in hits:
    print(f"  {h.chunk_id}  {h.text[:80]}")

## 9. Visualize

`fs.viz` builds typed graphs (`VizGraph`) at five abstraction levels and renders
them via pluggable backends. Requires the `[viz]` extra (`pip install -e '.[viz]'`).

| Level | Method                                  | What it shows |
| ----- | --------------------------------------- | ------------- |
| L0    | `fs.viz.entity_histogram(prefix=, k=)`  | Top-k entities by chunk_count (stats) |
| L1    | `fs.viz.entity_neighborhood(ontology_id)` | One entity + mentioning chunks + co-entities |
| L2    | `fs.viz.shelf(shelf_id)`                | One Layer A shelf + themes + chunks |
| L3    | `fs.viz.backbone(facet=)`               | Whole Layer A/B/C backbone |
| L4    | `fs.viz.ontology_subtree(ontology_id)`  | FoodOn ancestors + descendants |

Renderer choice via `.render(backend, output=...)`:
- `"pyvis"`      → interactive HTML (best in-notebook).
- `"cytoscape"`  → self-contained Cytoscape.js HTML, no Python deps beyond stdlib.
- `"graphviz"`   → static SVG / PNG (needs `dot` binary).
- `"matplotlib"` → bars / heatmaps over node weights (ignores edges; perfect for L0).

In [ ]:
# L0 — top entities, rendered with matplotlib (inline figure).
fig = fs.viz.entity_histogram(k=20).render("matplotlib")

In [ ]:
# L1 — entity neighborhood. Render to an HTML string and inline it via
# IPython.display.HTML(). Earlier we wrote the HTML to a file and embedded
# via IFrame(src=...) — VS Code's notebook webview can't fetch local files
# from a relative path, so the iframe stayed empty. Inline HTML works in
# every notebook environment.
from IPython.display import HTML

top = fs.entities.list(prefix='FOODON', k=1)
anchor_id = top[0].ontology_id if top else 'FOODON:03309927'

html = fs.viz.entity_neighborhood(anchor_id, max_chunks=8).render('cytoscape')
HTML(html)

In [ ]:
# L4 — ontology subtree around a single FoodOn id. Inline HTML, same
# recipe as L1.
from IPython.display import HTML

# Top entities by chunk_count are often leaves in the FoodOn hierarchy (e.g.
# `food calorie datum`) — picking one of those produces a 1-edge subtree.
# Walk the top-50 and prefer anchors with real downward structure: ≥ 3
# descendants ideally, then ≥ 1, then whatever's left.
_onto = fs.ontology
_top = [e.ontology_id for e in fs.entities.list(prefix='FOODON', k=50)]
_resolved = [(oid, len(_onto.id_to_descendants(oid)))
             for oid in _top if _onto.get(oid) is not None]
subtree_id = next((oid for oid, n in _resolved if n >= 3),
                  next((oid for oid, n in _resolved if n >= 1),
                       _resolved[0][0] if _resolved else anchor_id))
if subtree_id != anchor_id:
    n_desc = dict(_resolved).get(subtree_id, 0)
    print(f'L4 anchor: {subtree_id} ({n_desc} descendants) — L1 anchor {anchor_id} was a leaf or unresolved')

rg_subtree = fs.viz.ontology_subtree(subtree_id, max_descendants=20)
print(rg_subtree)

html = rg_subtree.render('cytoscape')
HTML(html)

In [ ]:
# L2 (Layer A backbone, foods facet). Inline HTML, same recipe as L1.
from IPython.display import HTML

rg_backbone = fs.viz.backbone(facet='foods')
print(rg_backbone)

html = rg_backbone.render('cytoscape')
HTML(html)

---

## Under the hood (appendix)

The three cells above are everything an end-user needs. The appendix below is
for contributors and shows the same pipeline **without** pre-computed NER/NEL —
i.e. running GLiNER + HNSW from scratch. Skip it unless you're debugging the
NER side.

### A1. Run GLiNER + HNSW directly

Requires `pip install -e '.[annotate]'`. First call downloads the GLiNER bio
model (~1.5 GB) and the BioLORD encoder (~440 MB), then builds and caches the
FoodOn HNSW index under `data/`. Subsequent runs are fast.

In [ ]:
import importlib.util as _u

needs = [pkg for pkg in ("gliner", "hnswlib", "sentence_transformers") if not _u.find_spec(pkg)]
if needs:
    print("Skipping — missing:", needs, "\nInstall with:  pip install -e '.[annotate]'")
else:
    fs_g = FoodScholar.from_config({
        "corpus": {"chunks_path": str(CORPUS_DIR)},
        "ontology": {
            "foodon_path": str(FOODON_OWL),
            "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
            "prefix_filter": ["FOODON:"],
        },
        "storage": {
            "chunk_store": {"backend": "memory"},
            "graph_store": {"backend": "memory"},
        },
    })
    one_file = next(iter(sorted(CORPUS_DIR.glob("*.csv"))))
    fs_g.ingest(one_file)  # no nel_dir → GLiNER + HNSW path
    c = fs_g.chunk_store.scan()[0]
    print(c.chunk_id, "mentions:", [m.text for m in c.mentions[:6]])

### A2. Inspect the graph

Layer B/C builders are still stubs — this appendix shows the `fs.graph` API
the builders will write through. Run it against the in-memory facade so it
doesn't depend on the live services.

In [ ]:
fs_local = FoodScholar.in_memory()
fs_local.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                         facet="dietary_patterns", depth=0)
fs_local.graph.add_theme(theme_id="t-olive", label="Olive oil benefits",
                         shelf_ids=["s-med"], discovered_by="leiden",
                         discovery_version="nb")
shelf = fs_local.graph.shelf("s-med")
print("shelf :", shelf.label, "| facet:", shelf.facet)
print("themes:", [t.label for t in shelf.themes()])